# Earlier CGB baseline (archived)

This self-contained notebook preserves the preceding experiment. Use [the current notebook](../../notebooks/cgb-conditional-paths.ipynb) for active research.

# CAD duration momentum — research notebook

24 September 2026 · Research design and mathematical companion to **cad-duration-momentum.ipynb**

The object is a live estimate of the direction, magnitude and adverse path of **CGB over the next 60, 120 and 240 minutes**. Canadian yields, US duration, swaps, forwards and liquidity describe the environment of that exposure. Buying or selling a cash bond instead of a future is a subsequent execution and risk-mapping decision.

The notebook contains the implementation, with its market-data inputs deliberately empty. Its equations and transformations are research choices, not estimates from Canadian data. It has no measured trading performance. It is a new CAD application informed by the source framework; it does not replace or silently rewrite his six notebooks.

The most important decision is this: **describe the state of the market, forecast an observable future, and keep those two objects separate.** Otherwise a model can become extremely good at predicting the persistence of its own labels.


Run in order. Empty inputs intentionally produce an **awaiting data** status. All implementation code is embedded here; no hidden local module is required. Dependencies: Python 3.12, NumPy, pandas, SciPy, XGBoost, and matplotlib for optional plots. This copy was checked with NumPy 2.0.2, pandas 2.2.2, SciPy 1.13.1 and XGBoost 3.0.3. CPU is the portable default; `xgb_device="cuda"` is an explicit option for a supported environment. This is a separate application, so the original repository's GPU-only deployment contract is unchanged.


## 1. What I take from the source author, and what the code actually does

His useful starting question is about the system's reference and the conditions under which its behavior remains stable. For this account, the exposure is already chosen: Canadian duration through CGB. We do not need to discover a basket before studying that exposure.

The original pipeline nevertheless gives us an ordering worth retaining:

| Original stage | Actual work | CAD translation |
|---|---|---|
| 1 — leader universe | Select baskets, weights and leaders within an exposure environment | Freeze CGB as the forecast object; declare contextual instruments and units |
| 2 — metrics | Estimate return shape, tails, entropy and state descriptors | Measure multiscale movement, variability, directional efficiency and liquidity observations causally |
| 3 — beta | Build a reference basket relationship and departures from it | Keep observed common US duration and Canadian residual movement separately visible |
| 4 — matrices | Specify seven structural transition laws over 22 states | Inspect descriptive phase transitions; use an explicit estimated persistence model as a separate forecasting expert |
| 5 — model | Construct current graph states, forecast future states with XGBoost, combine with structural and Monte Carlo components | Forecast actual future CGB outcomes; combine a drift expert and an ML expert without pretending they are independent evidence |
| 6 — signals | Read persisted forecasts and expose the state/action view | Produce a timestamped indicator, path estimates, disagreement and data status |

There are **three kinds of learning** here. Rolling covariances and distribution fits already estimate quantities from data. A state tagger then imposes a representation. Supervised ML learns a mapping from that representation and other features to future outcomes. Saying that learning only starts at XGBoost misses the first two choices.

In the original, the supervised target is a future graph tag. That can be a legitimate target: the future tag requires future observations. But a manually persistent tag grammar can make tag accuracy look impressive without establishing useful price prediction. Our primary supervised target is therefore a future price outcome. Current phase remains an explanatory feature and a diagnostic.

We retain the distinction between reference, disturbance, response and state. We change the original daily horizon, universe, state vocabulary and final combination deliberately. A daily 252-observation memory is not a justified 252-minute memory merely because the number is convenient.


## 2. The physics: useful structure, then an observation problem

The Smart Beta essay models capital as a distribution over exposures, not as the path of a single investor. In its notation,

$$
\int_\Omega \rho(w,t)\,dw=1,
\qquad
H(w,\rho,F)=-w\cdot F+\frac{\gamma}{2}\lVert w\rVert^2+\lambda(K*\rho)(w).
$$

Here $w$ is an exposure vector; $F$ favors some exposures; the quadratic term penalizes concentration; the interaction kernel represents how crowding affects other participants. The proposed individual and population dynamics are

$$
dw_t=-\nabla_w H\,dt+\sqrt{2D}\,dB_t,
$$

$$
\partial_t\rho=\nabla_w\cdot(\rho\nabla_w H)+D\Delta_w\rho.
$$

There is a substantive mathematical idea behind this. For fixed coefficients, a symmetric interaction kernel, suitable regularity and no flux through the boundary, define

$$
\begin{aligned}
\mathcal F[\rho]
&=\int\left(-w\cdot F+\frac{\gamma}{2}\lVert w\rVert^2\right)\rho(w)\,dw\\
&\quad+\frac{\lambda}{2}\iint K(w-w')\rho(w)\rho(w')\,dw\,dw'
+D\int\rho(w)\log\rho(w)\,dw.
\end{aligned}
$$

Writing the evolution as a gradient flow gives

$$
\frac{d\mathcal F}{dt}
=-\int\rho\left\lVert\nabla_w\frac{\delta\mathcal F}{\delta\rho}\right\rVert^2dw
\leq0.
$$

This calculation explains the attraction of the dissipation language. It does **not** establish that a CGB position earns the negative derivative of this functional. P&L has units of money and requires holdings and executable prices. A chosen free-energy functional has its own units and assumptions.

Nor is the inequality unconditional when the environment changes. If $F$ and $D$ vary with time, additional explicit-time terms include

$$
-\dot F_t\cdot\mathbb E_{\rho_t}[w]
+\dot D_t\int\rho_t\log\rho_t\,dw.
$$

Those terms need not be negative. This is precisely why changes in the surrounding constraints matter.

The hard step is the observation map. We do not observe the distribution of all investors' duration exposure, their funding constraints, or their remaining orders. Four days of swap quotes do not reveal those hidden objects. A full mean-field model would require an identified measurement equation and estimates for its interaction structure.

Our smaller, testable interpretation is: **directional adjustment may persist, observation noise may hide it, and the relationship between context and subsequent movement may change.** We estimate an observable consequence of that idea. The notebook calls its hidden variable *drift*, because price data can support an estimate of conditional movement; calling it physical selling pressure would add an identification claim that the data have not earned.

The same discipline applies to quantum language. The repository's final implementation combines nonnegative scores and normalizes them. An amplitude-and-square construction can be mathematically well defined without identifying financial observations with measured quantum amplitudes. We can study its forecasting behavior without importing a physical theorem as a financial probability guarantee.


### Configuration and feature vocabulary

These defaults define the first research version. None was optimized on market outcomes. `cgb` is the baseline. Add modules explicitly after reviewing their coverage; changing a module or threshold after seeing TEST creates a new experiment, not a stronger old result.


In [1]:
"""Causal measurements for the independent CAD duration research notebook.

No data is fabricated, no model is fitted here, and no order is sent. A snapshot
means what the receiver knew at its decision time, not the eventual corrected tape.
"""
from dataclasses import dataclass
import numpy as np
import pandas as pd


@dataclass(frozen=True)
class ResearchConfig:
    grid_minutes: int = 1
    horizons_minutes: tuple = (60, 120, 240)
    tick_size: float = .01
    tick_value_cad: float = 10.
    cgb_stale_seconds: float = 30.
    us_stale_seconds: float = 30.
    context_stale_seconds: float = 30.
    rates_stale_seconds: float = 300.
    vwap_stale_seconds: float = 300.
    volatility_window: int = 60
    beta_window: int = 120
    momentum_windows: tuple = (5, 15, 30, 60)
    phase_threshold: float = .75
    efficiency_threshold: float = .35
    neutral_band_sigma: float = .35
    feature_modules: tuple = ('cgb',)
    min_train_sessions: int = 10
    min_cal_sessions: int = 3
    min_test_sessions: int = 3
    embargo_minutes: int = 240
    xgb_device: str = 'cpu'
    random_state: int = 1729


RATE_INSTRUMENTS = (
    'CAD2Y', 'CAD5Y', 'CAD10Y', 'OIS1Y', 'OIS2Y', 'OIS3Y', 'OIS4Y', 'OIS5Y',
    'SWAP1Y', 'SWAP2Y', 'SWAP3Y', 'SWAP4Y', 'SWAP5Y',
    'FWD1Y1Y', 'FWD2Y1Y',
)
PHASE_NAMES = {
    -1: 'unready', 0: 'balance', 1: 'up-forming', 2: 'up-persistent',
    3: 'up-weakening', 4: 'down-forming', 5: 'down-persistent',
    6: 'down-weakening',
}
FEATURE_MODULES = {
    'cgb': [
        'sigma_ticks', 'mom_z_5', 'mom_z_15', 'mom_z_30', 'mom_z_60',
        'efficiency_30', 'range_position_30', 'vol_ratio_15_60',
        'return_skew_60', 'down_semivol_ratio_60', 'spread_ticks',
    ] + [f'phase_{name}' for name in PHASE_NAMES.values() if name != 'unready'],
    'us': [
        'us_move_5', 'us_move_30', 'beta_us_prior', 'common_ticks_1',
        'local_ticks_1', 'local_z_30', 'cad_us_corr_prior',
    ],
    'curve': [f'cad_{name}_bp_{w}' for name in ('level5', 'slope25', 'slope510')
              for w in (5, 30)],
    'ois': [f'{name.lower()}_change_bp_{w}' for name in ('OIS1Y', 'OIS2Y', 'OIS3Y', 'OIS4Y', 'OIS5Y')
            for w in (5, 30)],
    'swaps': [f'swap{year}y_change_bp_{w}' for year in range(1, 6) for w in (5, 30)],
    'cad_futures': [f'{name}_move_logbp_{w}' for name in ('cgz', 'cgf') for w in (5, 30)],
    'forwards': [f'{name.lower()}_change_bp_{w}' for name in ('FWD1Y1Y', 'FWD2Y1Y')
                 for w in (5, 30)],
    'book': ['book_imbalance', 'microprice_displacement_ticks',
             'book_imbalance_mean_5'],
    'vwap': ['vwap_distance_ticks', 'vwap_distance_sigma', 'vwap_std_ticks'],
    'context': ['spx_move_logbp_5', 'spx_move_logbp_30',
                'vix_move_points_5', 'vix_move_points_30'],
}



from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm
try:
    from IPython.display import display
except ImportError:
    display = print

cfg = ResearchConfig()
print("Feature modules:", tuple(FEATURE_MODULES))


Feature modules: ('cgb', 'us', 'curve', 'ois', 'swaps', 'cad_futures', 'forwards', 'book', 'vwap', 'context')


### Supply canonical data here — deliberately blank

| Input | Required columns | Units / identifiers |
|---|---|---|
| `QUOTES` | instrument, contract, event_time, available_at, bid, ask; optional bid_size, ask_size | Decimal price points; CGB, US10, CGZ, CGF; one deliberately selected contract stream per alias |
| `RATES` | instrument, event_time, available_at, rate_bp | Rate basis points: 3.25 percent must be 325; CAD2Y/5Y/10Y, OIS1Y–5Y, SWAP1Y–5Y, FWD1Y1Y/FWD2Y1Y |
| `SESSIONS` | session_id, open_time, close_time | Explicit permitted session, timezone-aware; unique IDs |
| `TRADES` | trade_id, instrument, contract, event_time, available_at, price, size | Optional CGB prints; unique IDs; positive trade size |
| `CONTEXT` | instrument, event_time, available_at, value | Optional SPX/VIX scalar marks in index points; no invented bid/ask |
| `AS_OF` | timezone-aware timestamp | The actual research cutoff or live decision time; independent of latest quote arrival |

All timestamps must be timezone-aware. Prices must already be decoded from vendor notation. Keep original revision availability. The simple trade adapter rejects corrections; an event-aware adapter must preserve their history instead of editing the original print retrospectively. No feed credentials or vendor format is assumed here.


In [2]:
QUOTES = None
RATES = None
SESSIONS = None
TRADES = None
CONTEXT = None
AS_OF = None

# Set these objects to real canonical pandas DataFrames and an aware timestamp.
# Example module selection AFTER confirming historical coverage:
# cfg = ResearchConfig(feature_modules=('cgb', 'us', 'curve', 'book'))
INPUTS_READY = QUOTES is not None and SESSIONS is not None and AS_OF is not None
print('Inputs supplied' if INPUTS_READY else 'Awaiting market data and an explicit AS_OF cutoff.')


Awaiting market data and an explicit AS_OF cutoff.


## 4. Time is part of the model

Each observation has an event time and a time when the model could first use it. The information set is

$$
\mathcal F_t=\sigma\{z_i:\operatorname{available\_at}_i\leq t\}.
$$

The notebook builds snapshots from the most recent event known at the decision time, using the latest available version of that event. A late revision of an old event cannot replace a newer market observation. Invalid and stale quotes remain invalid; the loader cannot rescue the row by silently falling back to an older attractive value.

The canonical inputs are atomic bid/ask quote records, rate records expressed in basis points, an explicit session schedule, optional trade records and optional scalar SPX/VIX marks. An explicit timezone-aware `AS_OF` cutoff identifies the current decision; the grid never extends into the future session. A vendor adapter is deliberately left to the empty input cells because no feed schema has been supplied. In particular, receipt timestamps must not be fabricated from exchange timestamps.

The session table is the account's permitted research/trading window. It is not an inferred exchange calendar. Prices must identify a contract. Rolling transformations reset when the session or contract changes; missing minutes stay in the grid. A switch between contracts is not a duration return.

This has a practical cost: a 120-minute reference estimate may be unavailable early in the session. The first implementation exposes that warmup instead of manufacturing a prior-session link. A later cross-session reference model would be a separate, explicit design change.

The forecast at $t$ uses completed information at $t$. It concerns movement after $t$. A trade benchmark uses the bid/ask available after its chosen latency. These are separate clocks.


### Validate clocks, units and identities


In [3]:
def _utc_column(frame, name):
    """Reject naive clocks: guessing a timezone would silently change causality."""
    values = frame[name]
    if values.isna().any():
        raise ValueError(f'{name} contains missing timestamps')
    if any(pd.Timestamp(value).tzinfo is None for value in values):
        raise ValueError(f'{name} must contain timezone-aware timestamps')
    return pd.to_datetime(values, utc=True)


def _validate_cfg(cfg):
    if not isinstance(cfg.grid_minutes, int) or cfg.grid_minutes <= 0:
        raise ValueError('grid_minutes must be a positive integer')
    if cfg.tick_size <= 0 or cfg.tick_value_cad <= 0:
        raise ValueError('Tick size/value must be positive')
    windows = {5, 15, 30, 60, cfg.volatility_window, cfg.beta_window,
               *cfg.momentum_windows, *cfg.horizons_minutes}
    if any(w <= 0 or w % cfg.grid_minutes for w in windows):
        raise ValueError('Every window/horizon must be positive and divisible by grid_minutes')
    if cfg.volatility_window / cfg.grid_minutes < 2 or cfg.beta_window / cfg.grid_minutes < 2:
        raise ValueError('Volatility and beta windows require at least two grid steps')
    if min(cfg.cgb_stale_seconds, cfg.us_stale_seconds, cfg.context_stale_seconds, cfg.rates_stale_seconds,
           cfg.vwap_stale_seconds) < 0:
        raise ValueError('Staleness limits cannot be negative')
    if cfg.phase_threshold < 0 or not 0 <= cfg.efficiency_threshold <= 1:
        raise ValueError('Invalid descriptive phase thresholds')
    if not np.isfinite(cfg.neutral_band_sigma) or cfg.neutral_band_sigma < 0:
        raise ValueError('neutral_band_sigma must be finite and nonnegative')
    unknown = set(cfg.feature_modules).difference(FEATURE_MODULES)
    if unknown:
        raise ValueError(f'Unknown feature modules: {sorted(unknown)}')


def _normalise_events(frame, fields, allowed):
    """Validate identity/clocks while retaining invalid numeric observations."""
    if frame is None or len(frame) == 0:
        return pd.DataFrame(columns=fields)
    frame = frame.copy()
    missing = set(fields).difference(frame.columns)
    if missing:
        raise ValueError(f'Input is missing columns: {sorted(missing)}')
    if frame['instrument'].isna().any():
        raise ValueError('Missing instrument identifier')
    unknown = set(frame['instrument']).difference(allowed)
    if unknown:
        raise ValueError(f'Unsupported instrument identifiers: {sorted(unknown)}')
    for clock in ('event_time', 'available_at'):
        frame[clock] = _utc_column(frame, clock)
    if (frame['event_time'] > frame['available_at']).any():
        raise ValueError('event_time exceeds available_at; repair the clock definition first')
    identity = ['instrument', 'event_time', 'available_at']
    if frame.duplicated(identity).any():
        raise ValueError('Ambiguous duplicate instrument/event/version timestamps')
    return frame.sort_values(['available_at', 'event_time'], kind='stable')


### Select the newest event that was actually known


In [4]:
def _known_snapshots(events, grid):
    """Newest event known, then newest version; late old events never rewind it.

    An invalid correction of the newest event remains the latest observation.
    Numeric validity is intentionally assessed only AFTER this temporal selection.
    One receiver timestamp is required per atomic bid/ask or rate snapshot.
    """
    if events.empty:
        return pd.DataFrame(index=grid)
    rows = list(events.to_dict('records'))
    output, position, newest = [], 0, None
    for decision in grid:
        while position < len(rows) and rows[position]['available_at'] <= decision:
            candidate = rows[position]
            if newest is None or candidate['event_time'] >= newest['event_time']:
                newest = candidate
            position += 1
        output.append({} if newest is None else newest.copy())
    return pd.DataFrame(output, index=grid)


def _quote_snapshot(events, grid, prefix, stale_seconds):
    snapshot = _known_snapshots(events, grid)
    output = pd.DataFrame(index=grid)
    for col in ('bid', 'ask', 'bid_size', 'ask_size'):
        output[f'{prefix}_{col}'] = pd.to_numeric(
            snapshot.get(col, pd.Series(np.nan, index=grid)), errors='coerce')
    output[f'{prefix}_contract'] = snapshot.get('contract', pd.Series(None, index=grid, dtype=object))
    for clock in ('event_time', 'available_at'):
        output[f'{prefix}_{clock}'] = pd.to_datetime(
            snapshot.get(clock, pd.Series(pd.NaT, index=grid)), utc=True)
    age = (grid.to_series() - output[f'{prefix}_event_time']).dt.total_seconds()
    output[f'{prefix}_age_seconds'] = age
    bid, ask = output[f'{prefix}_bid'], output[f'{prefix}_ask']
    valid = (np.isfinite(bid) & np.isfinite(ask) & (bid > 0) & (ask >= bid)
             & age.between(0, stale_seconds)
             & output[f'{prefix}_contract'].notna()
             & output[f'{prefix}_contract'].astype(str).str.len().gt(0))
    output[f'{prefix}_valid'] = valid
    output[f'{prefix}_mid'] = ((bid + ask) / 2).where(valid)
    return output


### Construct current snapshots and observed-trade VWAP


In [5]:
def prepare_panel(quotes, rates, sessions, cfg, trades=None, context=None, as_of=None):
    """Build an explicit receiver-time grid without crossing session/contract gaps.

    quotes: atomic CGB/US10/CGZ/CGF or quoted-context snapshots; rates: rate_bp.
    context: optional SPX/VIX scalar value marks with event/availability clocks. Prices
    must be decimal price points (convert Treasury fractional notation upstream).
    sessions: explicit open/close timestamps and unique session_id. No exchange
    calendar, contract-roll convention, vendor mapping or data revision is inferred.
    as_of is an explicit timezone-aware receiver cutoff, required for both live
    and historical runs. It is never inferred from the final source quote.
    """
    _validate_cfg(cfg)
    if as_of is None:
        raise ValueError('Set an explicit timezone-aware as_of receiver cutoff before constructing the panel')
    cutoff = pd.Timestamp(as_of)
    if pd.isna(cutoff) or cutoff.tzinfo is None:
        raise ValueError('as_of must be a nonmissing timezone-aware timestamp')
    cutoff = cutoff.tz_convert('UTC')
    if quotes is None or sessions is None:
        raise ValueError('Provide quotes and explicit sessions; no data is fabricated')
    quote_fields = ['instrument', 'contract', 'event_time', 'available_at', 'bid', 'ask']
    q = _normalise_events(quotes, quote_fields, {'CGB', 'US10', 'CGZ', 'CGF', 'SPX', 'VIX'})
    r = _normalise_events(rates, ['instrument', 'event_time', 'available_at', 'rate_bp'],
                          set(RATE_INSTRUMENTS))
    context_rows = _normalise_events(context, ['instrument', 'event_time', 'available_at', 'value'],
                                     {'SPX', 'VIX'})
    if set(context_rows['instrument']).intersection(set(q['instrument'])):
        raise ValueError('Choose scalar context OR quote context per instrument, not both')
    if q.empty or not q['instrument'].eq('CGB').any():
        raise ValueError('At least one CGB quote is required')
    required = {'session_id', 'open_time', 'close_time'}
    if not required.issubset(sessions.columns):
        raise ValueError(f'Sessions require {sorted(required)}')
    sessions = sessions.copy()
    if sessions['session_id'].isna().any() or sessions['session_id'].astype(str).duplicated().any():
        raise ValueError('session_id must be present and unique')
    for col in ('open_time', 'close_time'):
        sessions[col] = _utc_column(sessions, col)
    sessions = sessions.sort_values('open_time')
    if (sessions['close_time'] <= sessions['open_time']).any():
        raise ValueError('Session close must be later than open')
    if (sessions['open_time'].iloc[1:].to_numpy() <=
            sessions['close_time'].iloc[:-1].to_numpy()).any():
        raise ValueError('Sessions may not overlap or share an endpoint')
    panels = []
    for s in sessions[sessions['open_time'] <= cutoff].itertuples(index=False):
        grid = pd.date_range(s.open_time, min(s.close_time, cutoff),
                             freq=f'{cfg.grid_minutes}min', name='decision_time')
        if not len(grid):
            continue
        p = pd.DataFrame(index=grid)
        p['session_id'], p['session_close'] = s.session_id, s.close_time
        # At a new session, only new-session event observations are eligible.
        # This avoids yesterday's quote being considered fresh at an early open.
        session_q = q[(q['event_time'] >= s.open_time) & (q['available_at'] <= s.close_time)]
        for instrument, prefix, limit in (
            ('CGB', 'cgb', cfg.cgb_stale_seconds), ('US10', 'us', cfg.us_stale_seconds),
            ('CGZ', 'cgz', cfg.cgb_stale_seconds), ('CGF', 'cgf', cfg.cgb_stale_seconds),
            ('SPX', 'spx', cfg.context_stale_seconds), ('VIX', 'vix', cfg.context_stale_seconds),
        ):
            snap = _quote_snapshot(session_q[session_q['instrument'].eq(instrument)],
                                   grid, prefix, limit)
            p = p.join(snap)
        for instrument, prefix in (('SPX', 'spx'), ('VIX', 'vix')):
            p[f'{prefix}_measurement_kind'] = 'quote_mid'
            if instrument not in set(context_rows['instrument']):
                continue
            scalar_events = context_rows[(context_rows['instrument'].eq(instrument)) &
                                         (context_rows['event_time'] >= s.open_time) &
                                         (context_rows['available_at'] <= s.close_time)]
            snap = _known_snapshots(scalar_events, grid)
            value = pd.to_numeric(snap.get('value', pd.Series(np.nan, index=grid)), errors='coerce')
            for clock in ('event_time', 'available_at'):
                p[f'{prefix}_{clock}'] = pd.to_datetime(
                    snap.get(clock, pd.Series(pd.NaT, index=grid)), utc=True)
            age = (grid.to_series() - p[f'{prefix}_event_time']).dt.total_seconds()
            valid = np.isfinite(value) & (value > 0) & age.between(0, cfg.context_stale_seconds)
            p[f'{prefix}_age_seconds'], p[f'{prefix}_valid'] = age, valid
            p[f'{prefix}_mid'] = value.where(valid)
            p[f'{prefix}_contract'] = instrument + ':scalar_mark'
            p[f'{prefix}_measurement_kind'] = 'scalar_mark'
        p['contract'] = p['cgb_contract']
        contract_key = p['contract'].fillna('<unobserved>').astype(str)
        run = contract_key.ne(contract_key.shift()).cumsum()
        p['segment_id'] = str(s.session_id) + ':' + run.astype(str)
        for prefix in ('cgb', 'us', 'cgz', 'cgf', 'spx', 'vix'):
            same_contract = p[f'{prefix}_contract'].eq(p[f'{prefix}_contract'].shift())
            pair_valid = p[f'{prefix}_valid'] & p[f'{prefix}_valid'].shift(fill_value=False)
            changes = p[f'{prefix}_mid'].diff().where(same_contract & pair_valid)
            if prefix == 'cgb':
                p['cgb_ret_ticks'] = changes / cfg.tick_size
            elif prefix == 'vix':
                p['vix_ret_points'] = changes
            else:
                logret = 10000 * np.log(p[f'{prefix}_mid'] / p[f'{prefix}_mid'].shift())
                p[f'{prefix}_ret_logbp'] = logret.where(same_contract & pair_valid)
        session_r = r[(r['event_time'] >= s.open_time) & (r['available_at'] <= s.close_time)]
        for name in RATE_INSTRUMENTS:
            snapshot = _known_snapshots(session_r[session_r['instrument'].eq(name)], grid)
            rate = pd.to_numeric(snapshot.get('rate_bp', pd.Series(np.nan, index=grid)),
                                 errors='coerce')
            event_time = pd.to_datetime(snapshot.get('event_time', pd.Series(pd.NaT, index=grid)), utc=True)
            age = (grid.to_series() - event_time).dt.total_seconds()
            valid = np.isfinite(rate) & age.between(0, cfg.rates_stale_seconds)
            p[f'{name}_bp'], p[f'{name}_valid'] = rate.where(valid), valid
            p[f'{name}_age_seconds'] = age
        panels.append(p)
    if not panels:
        raise ValueError('No decision grid could be constructed')
    panel = pd.concat(panels).sort_index()
    if panel.index.has_duplicates:
        raise ValueError('Decision timestamps must be globally unique')
    return _attach_observed_vwap(panel, trades, cfg)


def _attach_observed_vwap(panel, trades, cfg):
    """Volume-weighted observed trade prices, with no inferred missing trades.

    Corrections/cancellations must be reconciled upstream into a point-in-time
    event adapter; this simple adapter rejects repeated trade IDs. Coverage of
    the venue trade tape cannot be certified from the supplied rows alone.
    """
    panel = panel.copy()
    for name in ('vwap', 'vwap_std', 'vwap_age_seconds', 'vwap_observed_volume'):
        panel[name] = np.nan
    panel['vwap_status'] = 'no_trades'
    panel['vwap_anchor'] = pd.Series(pd.NaT, index=panel.index, dtype='datetime64[ns, UTC]')
    if trades is None or len(trades) == 0:
        return panel
    required = {'trade_id', 'instrument', 'contract', 'event_time', 'available_at', 'price', 'size'}
    if not required.issubset(trades.columns):
        raise ValueError(f'Trades require {sorted(required)}')
    t = trades.copy()
    if t['trade_id'].isna().any() or t['trade_id'].duplicated().any():
        raise ValueError('Trade IDs must be unique; reconcile corrections before this adapter')
    if not t['instrument'].eq('CGB').all() or t['contract'].isna().any():
        raise ValueError('The trade adapter accepts identified CGB contracts only')
    for name in ('event_time', 'available_at'):
        t[name] = _utc_column(t, name)
    if (t['event_time'] > t['available_at']).any():
        raise ValueError('Trade event_time exceeds available_at')
    for name in ('price', 'size'):
        t[name] = pd.to_numeric(t[name], errors='coerce')
        if not (np.isfinite(t[name]) & t[name].gt(0)).all():
            raise ValueError(f'Trade {name} must be finite and positive')
    t = t.sort_values(['available_at', 'event_time'], kind='stable')
    for _, group in panel.groupby('segment_id', sort=False):
        contract = group['contract'].iloc[0]
        if pd.isna(contract):
            continue
        # Resets on each fixed-contract run. A late first quote means the anchor
        # is later than session open; display the anchor rather than invent data.
        anchor, endpoint = group.index[0], group.index[-1]
        panel.loc[group.index, 'vwap_anchor'] = anchor
        rows = t[(t['contract'].eq(contract)) & t['event_time'].between(anchor, endpoint)]
        rows = list(rows.to_dict('records'))
        j, volume, mean, m2, latest = 0, 0., 0., 0., None
        for decision in group.index:
            while j < len(rows) and rows[j]['available_at'] <= decision:
                trade = rows[j]
                new_volume = volume + trade['size']
                delta = trade['price'] - mean
                new_mean = mean + trade['size'] / new_volume * delta
                m2 += trade['size'] * delta * (trade['price'] - new_mean)
                mean, volume = new_mean, new_volume
                latest = trade['event_time'] if latest is None else max(latest, trade['event_time'])
                j += 1
            if volume <= 0:
                continue
            age = (decision - latest).total_seconds()
            panel.loc[decision, 'vwap_age_seconds'] = age
            panel.loc[decision, 'vwap_observed_volume'] = volume
            panel.loc[decision, 'vwap_status'] = 'observed_tape' if age <= cfg.vwap_stale_seconds else 'stale'
            if age <= cfg.vwap_stale_seconds:
                panel.loc[decision, 'vwap'] = mean
                panel.loc[decision, 'vwap_std'] = np.sqrt(max(m2 / volume, 0.))
    return panel


In [6]:
panel = None
if INPUTS_READY:
    panel = prepare_panel(QUOTES, RATES, SESSIONS, cfg, trades=TRADES, context=CONTEXT, as_of=AS_OF)
    display(panel.tail(3))
else:
    print('Panel unavailable: inputs remain blank.')


Panel unavailable: inputs remain blank.


## 5. Transform price into questions about movement

For each completed interval,

$$
r_t=\frac{m_t-m_{t-\Delta}}{\tau}.
$$

For several lookbacks $W$, we ask three different questions:

$$
M_{t,W}=\sum_{j=0}^{W-1}r_{t-j\Delta},
$$

$$
Z_{t,W}=\frac{M_{t,W}}{\widehat\sigma_t\sqrt W},
$$

$$
E_{t,W}=\frac{|M_{t,W}|}{\sum_{j=0}^{W-1}|r_{t-j\Delta}|},
\qquad 0\leq E_{t,W}\leq1.
$$

Raw movement says how many ticks changed. Scaled movement compares it with recent variability. Efficiency distinguishes a relatively direct path from travel that mostly cancels itself. A perfectly flat window has zero efficiency by convention; a missing window is missing, not flat.

These are not three independent witnesses. They are related views of the same path. The reason to keep them is that a five-tick decline through repeated failed rebounds differs from a five-tick decline in one jump followed by silence.

The initial lookbacks are 5, 15, 30 and 60 minutes. These are research resolutions around the intended holding horizon, not discovered optimal constants. Their point is to separate a recent impulse from established movement. Comparing horizons is preferable to assuming that a fast signal remains valid for four hours.

The volatility denominator uses preceding returns, with a declared floor to prevent numerical explosions. It does not use the future outcome window. Floors, lookbacks and phase thresholds are stored in configuration so they cannot disappear into unreviewable notebook state.

A useful objection is that normalization can make a tiny move in a quiet market look enormous. That is why the model retains both tick movement and its scale. Another objection is that price-derived variables may merely restate recent direction. The persistence baseline later asks exactly that question.

## 6. A reference frame that does not remove the trade

The Canadian movement can have an imported component and a local deviation. With comparable, available intervals, estimate a reference coefficient using prior paired returns:

$$
\widehat\beta_{t^-}
=\frac{\widehat{\operatorname{Cov}}_{t^-}(r^{C},u^{US})}
{\widehat{\operatorname{Var}}_{t^-}(u^{US})}.
$$

In this implementation $r^C$ is CGB ticks and $u^{US}$ is a US futures log-price change multiplied by $10^4$. The latter is **log-price basis points, not yield basis points**. Accordingly, beta has units of CGB ticks per US log-price basis point.

The observed decomposition is

$$
r^C_t=\underbrace{\widehat\beta_{t^-}u^{US}_t}_{\text{reference component}}
+\underbrace{\left(r^C_t-\widehat\beta_{t^-}u^{US}_t\right)}_{\text{local residual}}.
$$

Both terms remain features. We are forecasting outright CGB, so a US-led CGB selloff is still relevant. The residual is a diagnostic coordinate, not an instruction to hedge away the common movement.

The identity does not prove that the US market caused the Canadian move. Simultaneous reactions, stale observations and changes in hedge sensitivity are alternatives. Nor can a future US return appear on the right-hand side of a forecast unless that return itself is forecast. The notebook only feeds completed observed components into the predictor.

For the Canadian yield curve, use a lossless change of coordinates:

$$
L=\Delta y_5,\qquad S_{25}=\Delta y_5-\Delta y_2,
\qquad S_{5,10}=\Delta y_{10}-\Delta y_5.
$$

$$
\Delta y_2=L-S_{25},\qquad
\Delta y_5=L,\qquad
\Delta y_{10}=L+S_{5,10}.
$$

$L$ is a five-year anchor, not an assertion that we extracted a pure statistical level factor. This coordinate system lets us distinguish outright movement from rotation without inventing three independent signals out of three algebraically related curves.

For example, 2s5s flattening can occur because two-year yields rise faster than five-year yields, because five-year yields fall faster, or because one rises while the other falls. The same spread change cannot, by itself, establish relief for a CGB short. We preserve the component movements so the model can distinguish these cases.

Swaps and forwards extend that reference environment. A forward derived from the same discount curve is a transformation of existing information, not another independent vote. OIS-versus-government movements also include basis and instrument differences. They are not automatically clean observations of monetary-policy expectations.

## 7. Liquidity and VWAP: observable response, not a story about a firm

An aggregate depth snapshot can answer what was displayed at that moment. It does not reveal every cancellation, execution, hidden order or queue position.

At the best bid and ask, let displayed sizes be $q_b,q_a$. Two optional measurements are

$$
I_t=\frac{q_b-q_a}{q_b+q_a},
\qquad
\mu_t=\frac{a_tq_b+b_tq_a}{q_b+q_a}.
$$

$I_t$ is displayed imbalance. $\mu_t-m_t$ is the displacement of a size-weighted quote reference from the midpoint. These are best-quote measurements, not proof of actual aggressor flow, and not a calibrated probability of the next price move.

The first notebook version uses snapshots honestly. Reconstructing order-flow imbalance from events, cancellation intensity, replenishment or queue survival requires a defined event feed. That adapter and its exchange sequence semantics cannot be filled in reliably from a statement that L2 exists. The design records that boundary instead of coding invented event semantics.

When actual trade records are supplied, VWAP from the displayed fixed-contract anchor is

$$
\operatorname{VWAP}_t=\frac{\sum_{i:\,a_i\leq t}v_ip_i}{\sum_{i:\,a_i\leq t}v_i},
$$

where only received trades from the displayed anchor and contract enter. If the first known contract begins after session open, this anchor is later than the open; the output exposes that distinction rather than calling a partial tape the full session. A corresponding volume-weighted price dispersion is

$$
s^2_{V,t}=\frac{\sum v_i(p_i-\operatorname{VWAP}_t)^2}{\sum v_i}.
$$

This is a dispersion band, not a standard error or a universal probability interval. Its construction must match the chart being discussed before “one standard deviation” means the same thing on both screens.

The notebook can observe distance from VWAP and that dispersion reference. It does not define momentum as touching a line. A failed retest would be a later event label whose confirmation time must be recorded; a model cannot know at the first touch that the retest will fail. Such a pattern can be added as an explicit event detector after its rules are agreed, without becoming the whole definition of momentum.

The informative question is: after a recovery attempt, did price regain ground, did that recovery persist, and was the surrounding duration environment supportive? Identifying who supposedly traded is unnecessary for those measurements. A named-account anecdote can motivate a hypothesis; it cannot supply its probability.

## 8. Describe phase without making it destiny

The notebook uses seven descriptive phases: balanced, upward forming, upward persistent, upward weakening, and the corresponding three downward phases. Invalid or insufficient observations have a separate code.

Direction comes from scaled 30-minute movement. Persistence versus formation uses directional efficiency. Weakening uses the reduction of aligned recent momentum. The implementation makes the precedence and thresholds explicit.

These are a compact description of current observations. They are not seven proven natural kinds. They are also not the three supervised outcome classes: an upward-weakening phase can be followed by an upward, neutral or downward one-hour outcome.

An empirical transition matrix summarizes the description:

$$
\widehat P_{ij}=\frac{N_{ij}+\alpha}{\sum_kN_{ik}+K\alpha},\qquad K=7.
$$

Counts use only eligible successive observations in TRAIN. Additive smoothing prevents a never-seen transition from being declared physically impossible. The matrix is a diagnostic in this implementation; it is not secretly fed into the final forecast as another independent prior.

This is a deliberate departure from the original structural matrices. Its benefit is that we can ask whether apparent state persistence was created by thresholds and overlapping windows. Its cost is that we have not implemented the source framework's complete 22-state theory as a CAD market law. Copying that theory without Canadian identification would hide the most consequential research decision.


In [7]:
def _complete_change(series, steps):
    """Endpoint change only when every point in the interval is observed."""
    complete = series.notna().rolling(steps + 1, min_periods=steps + 1).sum().eq(steps + 1)
    return series.diff(steps).where(complete)


def _segment_features(p, cfg):
    """All rolling operations are confined to one session/contract run."""
    f = pd.DataFrame(index=p.index)
    steps = lambda minutes: int(minutes // cfg.grid_minutes)
    returns = p['cgb_ret_ticks']
    vol_n = steps(cfg.volatility_window)
    # This scale precedes the current completed return; 0.25 is a declared floor,
    # not a claim about exchange ticks or a fitted market noise parameter.
    f['sigma_ticks'] = returns.rolling(vol_n, min_periods=vol_n).std(ddof=1).shift(1).clip(lower=.25)
    windows = sorted({5, 15, 30, 60, *cfg.momentum_windows})
    for minutes in windows:
        n = steps(minutes)
        net = returns.rolling(n, min_periods=n).sum()
        path = returns.abs().rolling(n, min_periods=n).sum()
        f[f'mom_ticks_{minutes}'] = net
        f[f'mom_z_{minutes}'] = net / (f['sigma_ticks'] * np.sqrt(n))
        # A truly unchanged path has efficiency zero; an unobserved one stays NaN.
        f[f'efficiency_{minutes}'] = (net.abs() / path.where(path > 0)).where(path.ne(0), 0.)
    n30, n15, n60 = steps(30), steps(15), steps(60)
    mid = p['cgb_mid']
    low = mid.rolling(n30 + 1, min_periods=n30 + 1).min()
    high = mid.rolling(n30 + 1, min_periods=n30 + 1).max()
    width = high - low
    f['range_position_30'] = ((mid - low) / width.where(width > 0)).where(width.ne(0), .5)
    sigma15 = returns.rolling(n15, min_periods=n15).std(ddof=1)
    sigma60 = returns.rolling(n60, min_periods=n60).std(ddof=1)
    f['vol_ratio_15_60'] = sigma15 / sigma60.clip(lower=.25)
    f['return_skew_60'] = returns.rolling(n60, min_periods=n60).skew()
    energy = returns.pow(2).rolling(n60, min_periods=n60).sum()
    downside = returns.clip(upper=0).pow(2).rolling(n60, min_periods=n60).sum()
    f['down_semivol_ratio_60'] = (downside / energy.where(energy > 0)).where(energy.ne(0), .5)
    f['spread_ticks'] = ((p['cgb_ask'] - p['cgb_bid']) / cfg.tick_size).where(p['cgb_valid'])

    us = p['us_ret_logbp']
    n_beta = steps(cfg.beta_window)
    # Keep the regular grid, so a missing pair cannot be skipped in the window.
    cad_pair = returns.where(us.notna())
    us_pair = us.where(returns.notna())
    covariance = cad_pair.rolling(n_beta, min_periods=n_beta).cov(us_pair).shift(1)
    variance = us_pair.rolling(n_beta, min_periods=n_beta).var().shift(1)
    f['beta_us_prior'] = covariance / variance.where(variance > 1e-14)
    f['cad_us_corr_prior'] = cad_pair.rolling(n_beta, min_periods=n_beta).corr(us_pair).shift(1)
    f['common_ticks_1'] = f['beta_us_prior'] * us
    f['local_ticks_1'] = returns - f['common_ticks_1']
    local = f['local_ticks_1']
    local_sigma = local.rolling(vol_n, min_periods=vol_n).std().shift(1).clip(lower=.25)
    f['local_z_30'] = local.rolling(n30, min_periods=n30).sum() / (local_sigma * np.sqrt(n30))
    for minutes in (5, 30):
        n = steps(minutes)
        f[f'us_move_{minutes}'] = us.rolling(n, min_periods=n).sum()
        changes = {name: _complete_change(p[f'{name}_bp'], n) for name in RATE_INSTRUMENTS}
        f[f'cad_level5_bp_{minutes}'] = changes['CAD5Y']
        f[f'cad_slope25_bp_{minutes}'] = changes['CAD5Y'] - changes['CAD2Y']
        f[f'cad_slope510_bp_{minutes}'] = changes['CAD10Y'] - changes['CAD5Y']
        for name in RATE_INSTRUMENTS:
            f[f'{name.lower()}_change_bp_{minutes}'] = changes[name]
        f[f'spx_move_logbp_{minutes}'] = p['spx_ret_logbp'].rolling(n, min_periods=n).sum()
        f[f'vix_move_points_{minutes}'] = p['vix_ret_points'].rolling(n, min_periods=n).sum()
        for prefix in ('cgz', 'cgf'):
            f[f'{prefix}_move_logbp_{minutes}'] = p[f'{prefix}_ret_logbp'].rolling(n, min_periods=n).sum()

    bid_q = pd.to_numeric(p['cgb_bid_size'], errors='coerce')
    ask_q = pd.to_numeric(p['cgb_ask_size'], errors='coerce')
    size_ok = np.isfinite(bid_q) & np.isfinite(ask_q) & (bid_q >= 0) & (ask_q >= 0)
    total = (bid_q + ask_q).where(size_ok & p['cgb_valid'] & ((bid_q + ask_q) > 0))
    f['book_imbalance'] = (bid_q - ask_q) / total
    micro = (p['cgb_ask'] * bid_q + p['cgb_bid'] * ask_q) / total
    f['microprice_displacement_ticks'] = (micro - mid) / cfg.tick_size
    f['book_imbalance_mean_5'] = f['book_imbalance'].rolling(steps(5), min_periods=steps(5)).mean()
    f['vwap_distance_ticks'] = (mid - p['vwap']) / cfg.tick_size
    f['vwap_std_ticks'] = p['vwap_std'] / cfg.tick_size
    # Flat supplied trades have zero dispersion: a sigma-distance is undefined.
    f['vwap_distance_sigma'] = (mid - p['vwap']) / p['vwap_std'].where(p['vwap_std'] > 0)

    z, efficiency = f['mom_z_30'], f['efficiency_30']
    fast = f['mom_ticks_5'] / steps(5)
    slow = f['mom_ticks_30'] / n30
    direction = np.sign(z)
    active = z.abs() >= cfg.phase_threshold
    # "Weakening" is relative observed drift, never an inferred future reversal.
    weakening = direction * fast < .5 * direction * slow
    persistent = efficiency >= cfg.efficiency_threshold
    phase = pd.Series(0, index=p.index, dtype='int64')
    phase.loc[active & (direction > 0)] = 1
    phase.loc[active & (direction < 0)] = 4
    phase.loc[active & persistent & (direction > 0)] = 2
    phase.loc[active & persistent & (direction < 0)] = 5
    phase.loc[active & weakening & (direction > 0)] = 3
    phase.loc[active & weakening & (direction < 0)] = 6
    core_before_phase = FEATURE_MODULES['cgb'][:11]
    ready = np.isfinite(f[core_before_phase]).all(axis=1) & p['cgb_valid']
    phase.loc[~ready] = -1
    f['phase_code'] = phase
    for code, name in PHASE_NAMES.items():
        if code >= 0:
            f[f'phase_{name}'] = phase.eq(code).astype(float).where(ready)
    # Missing current CGB input invalidates the whole observation, not just price.
    f.loc[~p['cgb_valid'], f.columns.difference(['phase_code'])] = np.nan
    return f.replace([np.inf, -np.inf], np.nan)


def build_features(panel, cfg):
    """Return causal, unscaled features; module activation is an explicit choice.

    Core phase labels describe current price paths. Optional measurements remain
    NaN when unavailable. No rolling window, normalizer or phase uses future data.
    Recompute on a prefix and its features equal that prefix of the full run.
    """
    _validate_cfg(cfg)
    if panel.empty or not panel.index.is_monotonic_increasing or panel.index.has_duplicates:
        raise ValueError('Panel must be nonempty, sorted and uniquely indexed')
    result = pd.concat([_segment_features(group, cfg)
                        for _, group in panel.groupby('segment_id', sort=False)])
    return result.reindex(panel.index)


In [8]:
features = build_features(panel, cfg) if panel is not None else None
if features is not None:
    coverage_rows = []
    for name, columns in FEATURE_MODULES.items():
        complete = np.isfinite(features[columns]).all(axis=1)
        coverage_rows.append({'module': name, 'complete_rows': int(complete.sum()),
                              'sessions_with_complete_rows': panel.loc[complete, 'session_id'].nunique(),
                              'activated': name in cfg.feature_modules})
    display(pd.DataFrame(coverage_rows))
    display(features.tail(3))
else:
    print('Features unavailable: no market panel.')


Features unavailable: no market panel.


## 3. Define the future before inventing features

Let $m_t$ be the midpoint of one fixed CGB contract, and let $\tau=0.01$ be its price tick. The endpoint outcome is

$$
Y_{t,h}=\frac{m_{t+h}-m_t}{\tau}.
$$

A positive outcome benefits long duration through that future. This quantity is in CGB ticks, not yield basis points and not net profit.

We also retain adverse paths in both directions:

$$
A^+_{t,h}=\max_{0\leq u\leq h}\left[-\frac{m_{t+u}-m_t}{\tau}\right]_+,
\qquad
A^-_{t,h}=\max_{0\leq u\leq h}\left[\frac{m_{t+u}-m_t}{\tau}\right]_+.
$$

$A^+$ is adverse excursion for a long; $A^-$ is adverse excursion for a short. They are different targets, not the negative of each other. A lower endpoint after a large rally can be directionally correct and unpleasant to hold.

The implementation uses a regular observation grid. Its excursion labels measure the worst **observed grid midpoint**, so they can miss a more extreme move between grid points. They must not be described as exact tick-level excursions or guaranteed stop requirements.

For a three-class forecast, define a neutral band using information known now:

$$
b_{t,h}=c\,\widehat\sigma_t\sqrt{n_h},
\qquad n_h=h/\Delta,
$$

$$
C_{t,h}=
\begin{cases}
0,&Y_{t,h}<-b_{t,h},\\
1,&|Y_{t,h}|\leq b_{t,h},\\
2,&Y_{t,h}>b_{t,h}.
\end{cases}
$$

$\Delta$ is the grid interval and $\widehat\sigma_t$ is a past-only one-step volatility estimate in ticks. The initial $c=0.35$ is an editable resolution convention. The square-root scaling defines comparable thresholds; it does not assert that actual four-hour returns are independent Gaussian minute returns.

This label asks whether future movement is materially directional relative to the current scale. It does not subtract the spread and call the remainder a market state. Endpoint and path regressions retain continuous outcomes alongside these classes.

A target is absent if its full horizon crosses the explicit session close, a contract change, or missing/invalid observations. We do not turn a late-session four-hour target into a shorter target without changing its name.


## 13. The evaluation has specific objections to answer

We do not need a pile of disconnected significance tests. We need comparisons that could invalidate the reason for adding each layer.

| Claim being made | Comparison that can defeat it |
|---|---|
| Recent movement contains directional information | Compare against a zero-drift volatility-scaled forecast and simple momentum baseline |
| The drift model improves on raw persistence | Compare its held-out probability loss against the simpler momentum forecast |
| Nonlinear context matters | Compare XGBoost with the simple baseline and with the drift expert on the same eligible times |
| Combining experts is useful | Compare the pool against both components separately on untouched TEST |
| A new module adds information | Freeze its definition, then compare matched-period versions with and without it |
| Forecasted adverse excursion is meaningful | Inspect held-out pinball loss, realized coverage and error by session |
| The indicator is usable for trading | Only after the above, apply a declared position rule and price-based execution assumptions |

TRAIN estimates transforms that require fitting and learns model parameters. CAL fits the pool's weight and temperature. TEST supplies the final untouched comparison. Splits are chronological whole sessions, with explicit embargo and horizon eligibility; forward outcomes do not cross split boundaries.

Targets from neighboring minutes overlap. Thousands of one-minute rows do not become thousands of independent one-to-four-hour experiments. Reports therefore retain per-session losses and sample counts. A minimum number of sessions is an engineering gate, not a theorem of statistical adequacy.

For three-class probabilities, the two primary losses are

$$
\operatorname{LogLoss}=-\frac1N\sum_i\log p_i(C_i),
$$

$$
\operatorname{Brier}=\frac1N\sum_i\sum_{k=0}^2
\left(p_i(k)-\mathbf1_{\{C_i=k\}}\right)^2.
$$

The Brier convention here sums across classes without dividing by three. Smaller values are better for both losses. Accuracy alone ignores how strongly wrong forecasts were expressed.

The first serious result could be that the simple baseline matches the elaborate model. That would mean the extra structure has not earned its complexity on the available data. A second serious result could be that swaps help on four sessions but the evidence is too narrow to generalize. Neither result should be disguised by selecting a flattering chart.


In [9]:
def make_targets(panel, features, cfg):
    """Future midpoint outcomes; every grid observation through expiry is required."""
    out = {}
    if panel is None or len(panel) == 0:
        return out
    for horizon in cfg.horizons_minutes:
        if horizon % cfg.grid_minutes:
            raise ValueError("Every horizon must be a multiple of grid_minutes.")
        n = horizon // cfg.grid_minutes
        y = pd.DataFrame(index=panel.index)
        for col in ("return_ticks", "mae_long_ticks", "mae_short_ticks", "class_id"):
            y[col] = np.nan
        y["target_end"] = pd.Series(pd.NaT, index=y.index, dtype="datetime64[ns, UTC]")
        y["neutral_band_ticks"] = cfg.neutral_band_sigma * features.sigma_ticks * np.sqrt(n)
        for _, group in panel.groupby("segment_id", sort=False):
            idx = group.index
            mid = group.cgb_mid.where(group.cgb_valid)
            future_min = mid.iloc[::-1].rolling(n + 1, min_periods=n + 1).min().iloc[::-1]
            future_max = mid.iloc[::-1].rolling(n + 1, min_periods=n + 1).max().iloc[::-1]
            last = mid.shift(-n)
            end = pd.Series(idx, index=idx).shift(-n)
            band = y.loc[idx, "neutral_band_ticks"]
            good = (future_min.notna() & last.notna() & band.notna()
                    & ((end - pd.Series(idx, index=idx)) == pd.Timedelta(minutes=horizon))
                    & (end <= group.session_close))
            ret = (last - mid) / cfg.tick_size
            y.loc[idx, "return_ticks"] = ret.where(good)
            y.loc[idx, "mae_long_ticks"] = ((mid - future_min) / cfg.tick_size).clip(lower=0).where(good)
            y.loc[idx, "mae_short_ticks"] = ((future_max - mid) / cfg.tick_size).clip(lower=0).where(good)
            cls = pd.Series(np.where(ret < -band, 0, np.where(ret > band, 2, 1)), index=idx)
            y.loc[idx, "class_id"] = cls.where(good)
            y.loc[idx, "target_end"] = end.where(good)
        out[int(horizon)] = y
    return out


def split_sessions(panel, cfg):
    """One chronological holdout; complete sessions plus explicit boundary embargo."""
    empty = pd.DatetimeIndex([], tz="UTC", name="decision_time")
    if panel is None or len(panel) == 0:
        return {"train": empty, "cal": empty, "test": empty, "metadata": {"status": "no_data"}}
    ordered = panel.groupby("session_id", sort=False).apply(
        lambda x: x.index.min(), include_groups=False).sort_values().index.tolist()
    need = cfg.min_train_sessions + cfg.min_cal_sessions + cfg.min_test_sessions
    if len(ordered) < need:
        return {"train": empty, "cal": empty, "test": empty,
                "metadata": {"status": "insufficient_history", "sessions": len(ordered), "required": need}}
    nt, nc = cfg.min_test_sessions, cfg.min_cal_sessions
    sessions = {"train": ordered[:-(nt + nc)], "cal": ordered[-(nt + nc):-nt], "test": ordered[-nt:]}
    splits = {name: panel.index[panel.session_id.isin(ids)] for name, ids in sessions.items()}
    removed, cutoffs = {}, {}
    for left, right in (("train", "cal"), ("cal", "test")):
        cutoff = splits[right].min() - pd.Timedelta(minutes=cfg.embargo_minutes)
        before = len(splits[left])
        splits[left] = splits[left][splits[left] < cutoff]
        removed[left], cutoffs[left] = before - len(splits[left]), str(cutoff)
    splits["metadata"] = {"status": "ready", "sessions": sessions,
                          "embargo_minutes": cfg.embargo_minutes,
                          "embargo_rows_removed": removed, "cutoffs": cutoffs,
                          "purge_rule": "A label's entire horizon must end inside its own retained partition."}
    return splits


def _session_weights(session_ids):
    """Each included session contributes equal total mass; no class rebalancing."""
    s = pd.Series(np.asarray(session_ids))
    weights = 1.0 / s.map(s.value_counts()).to_numpy(dtype=float)
    return weights / weights.sum() * len(weights)


In [10]:
targets = make_targets(panel, features, cfg) if panel is not None else {}
splits = split_sessions(panel, cfg)
display(splits['metadata'])
if targets:
    display(pd.DataFrame([{'horizon_minutes': h, 'eligible_outcomes': y.class_id.notna().sum()}
                           for h, y in targets.items()]))


{'status': 'no_data'}


## 9. Implement a model of continuing adjustment

The first forecasting expert is a local drift model. It separates a slowly changing directional tendency from immediate price noise:

$$
d_t=\phi d_{t-\Delta}+\eta_t,
\qquad \eta_t\sim\mathcal N(0,Q),
$$

$$
r_t=d_t+\epsilon_t,
\qquad \epsilon_t\sim\mathcal N(0,R).
$$

$d_t$ has units of expected ticks per grid interval. $Q$ describes variation in the latent tendency. $R$ describes observation variation around it. This is a specified reduced model, not a derivation of price from the full exposure-density equation.

The training procedure estimates $\phi,Q,R$ from TRAIN observations by equal-session-weighted Gaussian innovation likelihood. Parameters then remain frozen through calibration and test. The filter can update its current estimate as new returns arrive; that is causal state updating, not refitting against the eventual test result.

For filtered mean $\widehat d_t$ and variance $P_t$, the prediction and update are

$$
\widehat d_{t|t-\Delta}=\phi\widehat d_{t-\Delta},
\qquad P_{t|t-\Delta}=\phi^2P_{t-\Delta}+Q,
$$

$$
v_t=r_t-\widehat d_{t|t-\Delta},\qquad
S_t=P_{t|t-\Delta}+R,\qquad
K_t=P_{t|t-\Delta}/S_t,
$$

$$
\widehat d_t=\widehat d_{t|t-\Delta}+K_tv_t,
\qquad P_t=(1-K_t)P_{t|t-\Delta}.
$$

A large surprise changes the state according to how much uncertainty the model assigned to drift and observation noise. There is no hindsight smoother. A future price cannot revise the state that was available at 10:15.

The fitted likelihood minimizes, up to constants,

$$
\frac12\sum_{t\in\mathrm{TRAIN}}\omega_t\omega_t\left(\log S_t+\frac{v_t^2}{S_t}\right).
$$

The implemented parameter bounds $0\leq\phi\leq0.999$ express a decaying-persistence family. Negative autocorrelated drift and explosive drift lie outside that family. That is an explicit model restriction, not a claim those markets are impossible. ML and the baseline comparison can reveal situations in which the restriction is inadequate.

For $n$ future intervals, define

$$
A_n=\sum_{j=1}^n\phi^j,\qquad B_k=\sum_{j=0}^k\phi^j.
$$

Then the conditional terminal distribution under this model is

$$
Y_{t,h}\mid\mathcal F_t
\sim\mathcal N\left(A_n\widehat d_t,
A_n^2P_t+Q\sum_{k=0}^{n-1}B_k^2+nR\right).
$$

This equation shows exactly what persistence buys us. The current drift contributes repeatedly, but with decay. Uncertainty accumulates from the uncertain starting drift, future drift innovations, and future observation noise.

With $\phi=0$, the current drift has no predictive carry into the next interval. With $\phi$ near one, it matters for much longer. With a small $Q/R$, the filter treats isolated noisy prints cautiously. These mechanisms are inspectable rather than hidden behind the word momentum.

The Gaussian assumption is vulnerable to jumps, fat tails and changing volatility. Simulated paths from this expert are scenarios conditional on those assumptions. They are not evidence that actual CAD tails are Gaussian. The optional simulation cell runs only for a selected current forecast; it does not manufacture a historical performance sample.


In [11]:
@dataclass
class LocalDrift:
    phi: float
    q: float
    r: float
    optimizer_success: bool
    optimizer_message: str
    n_observations: int
    boundary_warning: bool


def _filter_drift(returns, segments, model):
    """Forward Kalman filter only. Missing returns reset state, never become zero."""
    means, variances = np.empty(len(returns)), np.empty(len(returns))
    innovation, innovation_var = np.full(len(returns), np.nan), np.full(len(returns), np.nan)
    mean, variance, previous = 0.0, 0.0, None
    for i, (value, segment) in enumerate(zip(returns, segments)):
        if previous is None or segment != previous:
            mean, variance = 0.0, model.q / max(1.0 - model.phi ** 2, 1e-6)
        else:
            mean, variance = model.phi * mean, model.phi ** 2 * variance + model.q
        if np.isfinite(value):
            residual, total = value - mean, variance + model.r
            gain = variance / total
            mean, variance = mean + gain * residual, (1.0 - gain) * variance
            innovation[i], innovation_var[i] = residual, total
        else:
            mean, variance = 0.0, model.q / max(1.0 - model.phi ** 2, 1e-6)
        means[i], variances[i], previous = mean, max(variance, 0.0), segment
    return means, variances, innovation, innovation_var


def fit_local_drift(panel_train):
    """Estimate phi,Q,R on training observations using equal-session Gaussian NLL."""
    returns = panel_train.cgb_ret_ticks.to_numpy(dtype=float)
    valid = np.isfinite(returns)
    if valid.sum() < 200:
        raise ValueError("At least 200 valid training returns are required for the dynamics expert.")
    scale = max(float(np.nanvar(returns)), 0.0625)
    weights = np.zeros(len(returns))
    weights[valid] = _session_weights(panel_train.session_id.to_numpy()[valid])
    segments = panel_train.segment_id.to_numpy()

    def objective(theta):
        model = LocalDrift(float(theta[0]), float(np.exp(theta[1])), float(np.exp(theta[2])), False, "", 0, False)
        _, _, error, variance = _filter_drift(returns, segments, model)
        losses = 0.5 * (np.log(2 * np.pi * variance[valid]) + error[valid] ** 2 / variance[valid])
        return float(np.average(losses, weights=weights[valid]))

    floor, ceiling = scale * 1e-6, scale * 100
    result = minimize(objective, [0.9, np.log(scale * .03), np.log(scale * .7)], method="L-BFGS-B",
                      bounds=[(0.0, .999), (np.log(floor), np.log(ceiling)), (np.log(floor), np.log(ceiling))],
                      options={"maxiter": 200, "ftol": 1e-9})
    phi, q, r = float(result.x[0]), float(np.exp(result.x[1])), float(np.exp(result.x[2]))
    if not result.success or not np.isfinite(result.fun):
        raise ValueError("Dynamics likelihood fit did not converge: " + str(result.message))
    boundary = phi < .001 or phi > .998 or min(q, r) <= floor * 1.1 or max(q, r) >= ceiling / 1.1
    return LocalDrift(phi, q, r, bool(result.success), str(result.message), int(valid.sum()), boundary)


def filter_local_drift(panel, model):
    means, variances, errors, noise = _filter_drift(panel.cgb_ret_ticks.to_numpy(dtype=float),
                                                  panel.segment_id.to_numpy(), model)
    return pd.DataFrame({"drift_mean": means, "drift_variance": variances,
                         "innovation": errors, "innovation_variance": noise}, index=panel.index)


def drift_terminal_moments(filtered, model, steps):
    """Distribution of sum r_(t+1)...r_(t+n), conditional on current filtered state."""
    powers = model.phi ** np.arange(1, steps + 1)
    initial_loading = float(powers.sum())
    shock_loadings = np.cumsum(model.phi ** np.arange(steps))
    variance = (initial_loading ** 2 * filtered.drift_variance.to_numpy()
                + model.q * np.dot(shock_loadings, shock_loadings) + steps * model.r)
    return initial_loading * filtered.drift_mean.to_numpy(), np.maximum(variance, 1e-12)


def _gaussian_classes(mean, std, band):
    down = norm.cdf((-band - mean) / std)
    up = norm.sf((band - mean) / std)
    return _normalize_probabilities(np.column_stack((down, np.maximum(1 - down - up, 0), up)))


def _normalize_probabilities(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1)
    return p / p.sum(axis=1, keepdims=True)


## 10. Where machine learning enters

The second expert uses XGBoost to learn how the currently observed configuration relates to the actual future outcome. For each horizon it learns

$$
p^{ML}_{t,h}(k)=\Pr(C_{t,h}=k\mid X_t),\qquad k\in\{0,1,2\}.
$$

The input $X_t$ contains only declared feature modules. The initial runnable baseline uses CGB features. US duration, the Canadian curve, the book, swaps, VWAP and context are explicit additions, each requiring real historical coverage. The data cells and module selection are separate so that four swap sessions cannot masquerade as a month of swap evidence.

The three-class model minimizes weighted cross-entropy plus tree regularization:

$$
\mathcal L=-\sum_i\omega_i\log p^{ML}_{i,h}(C_{i,h})+\Omega(\text{trees}).
$$

Weights give sessions equal total weight. They are not inverse-class weights. Reweighting classes would change the probability target; it is not an innocuous way of making the market more predictable.

Small fixed trees provide a starting capacity. They can learn interactions such as: a downward CGB move with a negative local residual and persistent curve repricing differs from the same CGB move during a supportive US reversal. We do not impose those illustrative signs as truths. The data can fail to support them.

Separate regression heads estimate endpoint movement and the 80th conditional percentile of adverse excursion for each direction. The excursion loss is pinball loss,

$$
\ell_q(y,\widehat y)
=(y-\widehat y)\left(q-\mathbf1_{\{y<\widehat y\}}\right),\qquad q=0.8.
$$

A conditional 80th-percentile estimate is not a risk limit or a guaranteed coverage statement. Its coverage and pinball loss must be checked on untouched outcomes. Sparse tail examples make it especially uncertain with a short history.

This is where ML earns its place: learning nonlinear combinations and horizon-specific associations that a fixed drift equation cannot represent. It does not get to redefine the target after observing the test period. The initial notebook does not conduct a search for whichever feature set and horizon happens to win.

## 11. Combine experts without counting a story twice

The drift expert and ML expert observe related price information. Their outputs are conditional forecasts, not independent measurements. Multiplying them and calling the result Bayes would require assumptions we do not have.

Instead, use a declared logarithmic opinion pool:

$$
z_k=w\log p^{ML}_{t,h}(k)+(1-w)\log p^{D}_{t,h}(k),
$$

$$
p^*_{t,h}(k)=\frac{\exp(z_k/T)}{\sum_j\exp(z_j/T)},
\qquad 0\leq w\leq1,\quad0.5\leq T\leq5.
$$

$w$ controls the combination. $T$ controls sharpness. Both are fitted on a chronologically later calibration block, never on TEST. Numerical probability floors avoid taking a logarithm of zero; they do not prove that an outcome is physically possible with that exact probability.

The convex weights have a useful property: if the two experts give the same probability vector and $T=1$, the pool returns that same vector. Agreement alone does not square the probabilities and invent additional certainty. Calibration may still overfit a short block, so the report preserves both experts separately.

This is our CAD combination, not the repository's Born/Zurek posterior. The source audit at the end describes the latter precisely. Substituting this pool is an explicit research choice: it lets a small amount of calibration data estimate two inspectable quantities while exposing dependence between the experts.

The live directional indicator is

$$
I_{t,h}=p^*_{t,h}(2)-p^*_{t,h}(0),\qquad -1\leq I_{t,h}\leq1.
$$

It is a directional probability contrast. It is not expected ticks, a contract quantity or a trade recommendation. A 60-minute bearish indicator and a 240-minute neutral indicator can coexist. Averaging them into one compelling color would erase useful disagreement.

## 12. Three uncertainties that should not share one label

The output preserves three separate objects:

| Object | What it tells us | What it does not tell us |
|---|---|---|
| Market-outcome probabilities | Down/neutral/up probabilities within the fitted model | Whether the model itself is correct |
| Expert disagreement and forecast entropy | Whether experts disagree and whether the distribution is concentrated | A complete posterior over model uncertainty |
| Observation and history status | Freshness, missing feeds, warmup, session eligibility and training coverage | That a well-observed market must be predictable |

Normalized forecast entropy is

$$
H_t=-\frac{\sum_{k=0}^2p_k\log p_k}{\log3}.
$$

A forecast concentrated on the neutral class is different from a diffuse forecast spread across all three outcomes. Both can have an indicator near zero. A missing feed is different again.

The notebook also reports disagreement between the two expert vectors. This does not replace epistemic uncertainty analysis. Parameter uncertainty, missing mechanisms and distribution changes remain outside a single three-number probability vector.

Fresh data with contradictory signals can be economically informative. Contradiction is not automatically a broken model. Conversely, stale data that look harmonious are not confirmation.


In [12]:
def opinion_pool(p_ml, p_dynamics, weight, temperature):
    """Dependent conditional forecasts; convex log pool, not likelihood multiplication."""
    logits = (weight * np.log(_normalize_probabilities(p_ml))
              + (1 - weight) * np.log(_normalize_probabilities(p_dynamics))) / temperature
    return np.exp(logits - logsumexp(logits, axis=1, keepdims=True))


def fit_opinion_pool(p_ml, p_dynamics, classes, weights):
    classes = np.asarray(classes, dtype=int)

    def objective(theta):
        p = opinion_pool(p_ml, p_dynamics, theta[0], np.exp(theta[1]))
        return float(np.average(-np.log(p[np.arange(len(classes)), classes]), weights=weights))

    result = minimize(objective, [.5, 0.0], method="L-BFGS-B", bounds=[(0, 1), (np.log(.5), np.log(5))])
    if not result.success or not np.isfinite(result.fun):
        raise ValueError("Calibration optimization did not converge: " + str(result.message))
    return {"weight_ml": float(result.x[0]), "temperature": float(np.exp(result.x[1])),
            "calibration_nll": float(result.fun), "optimizer_success": bool(result.success)}


def phase_transition_diagnostic(panel, features, train_index, smoothing=.5):
    """Descriptive train-only transitions, with adjacent same-segment pairs only."""
    counts = np.zeros((7, 7), dtype=int)
    allowed = panel.index.isin(train_index)
    codes, segments = features.phase_code.to_numpy(), panel.segment_id.to_numpy()
    for i in range(1, len(panel)):
        if (allowed[i - 1] and allowed[i] and segments[i - 1] == segments[i]
                and 0 <= codes[i - 1] < 7 and 0 <= codes[i] < 7):
            counts[int(codes[i - 1]), int(codes[i])] += 1
    prior = counts + smoothing
    probability = prior / prior.sum(axis=1, keepdims=True)
    return {"counts": counts, "probability": probability, "smoothing": smoothing,
            "use": "description only; not a target or a constraint on future price direction"}


def _metrics(p, classes, returns, mean_prediction, session_ids):
    classes = np.asarray(classes, dtype=int)
    weights = _session_weights(session_ids)
    truth = np.eye(3)[classes]
    losses = -np.log(np.clip(p[np.arange(len(classes)), classes], 1e-9, 1))
    brier = np.sum((p - truth) ** 2, axis=1)
    absolute = np.abs(np.asarray(returns) - np.asarray(mean_prediction))
    rows = pd.DataFrame({"session_id": np.asarray(session_ids), "log_loss": losses,
                         "brier": brier, "endpoint_abs_error_ticks": absolute})
    if hasattr(returns, "index"):
        rows.index = returns.index
    per_session = rows.groupby("session_id").mean()
    confidence, prediction = p.max(axis=1), p.argmax(axis=1)
    reliability = pd.DataFrame({"confidence": confidence, "correct": prediction == classes,
                                "weight": weights, "bin": np.minimum((confidence * 5).astype(int), 4)})
    bins = []
    for bin_id, group in reliability.groupby("bin"):
        bins.append({"bin": int(bin_id), "rows": len(group),
                     "mean_confidence": float(np.average(group.confidence, weights=group.weight)),
                     "accuracy": float(np.average(group.correct, weights=group.weight))})
    return {"log_loss": float(np.average(losses, weights=weights)),
            "brier": float(np.average(brier, weights=weights)),
            "endpoint_mae_ticks": float(np.average(absolute, weights=weights)),
            "rows": len(classes), "sessions": len(per_session),
            "per_session": per_session, "pointwise_losses": rows, "reliability": pd.DataFrame(bins)}


def _feature_columns(features, cfg):
    if "cgb" not in cfg.feature_modules:
        raise ValueError("The cgb feature module is mandatory.")
    selected = []
    for module in cfg.feature_modules:
        if module not in FEATURE_MODULES:
            raise ValueError(f"Unknown feature module {module!r}; choose from {tuple(FEATURE_MODULES)}")
        selected.extend(FEATURE_MODULES[module])
    selected = list(dict.fromkeys(selected))
    missing = set(selected).difference(features.columns)
    if missing:
        raise ValueError(f"Missing constructed feature columns: {sorted(missing)}")
    return selected


def _label_indices(y, candidate, core_valid):
    idx = y.index.intersection(candidate)
    good = (y.loc[idx, "class_id"].notna() & y.loc[idx, "target_end"].isin(candidate)
            & core_valid.loc[idx])
    return idx[good]


def _module_coverage(panel, features, train_index, cfg):
    coverage = {}
    for name in cfg.feature_modules:
        cols = FEATURE_MODULES[name]
        complete = np.isfinite(features.loc[train_index, cols]).all(axis=1)
        usable = train_index[complete]
        coverage[name] = {"rows": len(usable), "sessions": panel.loc[usable, "session_id"].nunique()}
    return coverage


### Fit the declared research version

The code below creates models only when all coverage and chronology checks pass. It fits no parameters to TEST and performs no search for a flattering module set. The pooled endpoint MAE column shares the separate ML mean head; the probability pool itself does not determine a continuous mean.


In [13]:
def fit_research(panel, features, targets, cfg):
    """Fit once on TRAIN, pool on CAL, report on TEST. No execution or auto-selection."""
    base = {"status": "no_data", "horizons": {}, "report": pd.DataFrame(), "splits": {}, "config": cfg}
    if panel is None or features is None or len(panel) == 0:
        return base
    splits = split_sessions(panel, cfg)
    base["splits"] = splits
    if splits["metadata"]["status"] != "ready":
        base.update(status="insufficient_history", reason=splits["metadata"])
        return base
    columns = _feature_columns(features, cfg)
    x = features[columns].replace([np.inf, -np.inf], np.nan)
    core = np.isfinite(x[FEATURE_MODULES["cgb"]]).all(axis=1) & panel.cgb_valid & (features.phase_code >= 0)
    coverage = _module_coverage(panel, features, splits["train"], cfg)
    base.update(feature_columns=columns, module_coverage=coverage)
    deficient = {name: value for name, value in coverage.items()
                 if value["sessions"] < cfg.min_train_sessions or value["rows"] < 100}
    if deficient:
        base.update(status="insufficient_history", reason={"module_training_coverage": deficient})
        return base
    try:
        from xgboost import XGBClassifier, XGBRegressor
    except ImportError:
        base.update(status="dependency_missing", reason="Install xgboost >= 2.0 for reg:quantileerror.")
        return base
    try:
        dynamics = fit_local_drift(panel.loc[splits["train"]])
    except ValueError as exc:
        base.update(status="dynamics_fit_failed", reason=str(exc))
        return base
    filtered = filter_local_drift(panel, dynamics)
    base.update(dynamics=dynamics, filtered_state=filtered, filter_history_start=panel.index[0],
                available_after=splits["cal"].max(),
                phase_transitions=phase_transition_diagnostic(panel, features, splits["train"]))
    settings = dict(n_estimators=100, max_depth=2, learning_rate=.03, min_child_weight=10,
                    subsample=1.0, colsample_bytree=1.0, reg_lambda=10.0,
                    tree_method="hist", device=cfg.xgb_device, random_state=cfg.random_state, n_jobs=2)
    reports = []
    for horizon in cfg.horizons_minutes:
        entry = {"status": "insufficient_history"}
        base["horizons"][int(horizon)] = entry
        if horizon not in targets:
            entry["reason"] = "Targets have not been constructed for this horizon."
            continue
        y = targets[horizon]
        used = {name: _label_indices(y, splits[name], core) for name in ("train", "cal", "test")}
        session_counts = {name: panel.loc[idx, "session_id"].nunique() for name, idx in used.items()}
        entry.update(indices=used, eligible_sessions=session_counts,
                     rows={name: len(idx) for name, idx in used.items()})
        minimum = {"train": cfg.min_train_sessions, "cal": cfg.min_cal_sessions, "test": cfg.min_test_sessions}
        if any(session_counts[name] < minimum[name] for name in minimum) or any(len(i) < 30 for i in used.values()):
            entry["reason"] = "Full-horizon valid outcomes do not cover the required sessions and at least 30 rows per partition."
            continue
        tr, ca, te = (used[name] for name in ("train", "cal", "test"))
        horizon_coverage = _module_coverage(panel, features, tr, cfg)
        entry["module_training_coverage"] = horizon_coverage
        if any(value["sessions"] < cfg.min_train_sessions or value["rows"] < 100 for value in horizon_coverage.values()):
            entry["reason"] = "An enabled module lacks training coverage on this horizon's eligible outcomes."
            continue
        if set(y.loc[tr, "class_id"].astype(int)) != {0, 1, 2}:
            entry.update(status="insufficient_outcome_coverage", reason="TRAIN must contain down, neutral, and up outcomes.")
            continue
        wt, wc = _session_weights(panel.loc[tr, "session_id"]), _session_weights(panel.loc[ca, "session_id"])
        clf = XGBClassifier(objective="multi:softprob", num_class=3, eval_metric="mlogloss", **settings)
        mean_model = XGBRegressor(objective="reg:squarederror", **settings)
        long_risk = XGBRegressor(objective="reg:quantileerror", quantile_alpha=.8, **settings)
        short_risk = XGBRegressor(objective="reg:quantileerror", quantile_alpha=.8, **settings)
        clf.fit(x.loc[tr], y.loc[tr, "class_id"].astype(int), sample_weight=wt)
        mean_model.fit(x.loc[tr], y.loc[tr, "return_ticks"], sample_weight=wt)
        long_risk.fit(x.loc[tr], y.loc[tr, "mae_long_ticks"], sample_weight=wt)
        short_risk.fit(x.loc[tr], y.loc[tr, "mae_short_ticks"], sample_weight=wt)
        eligible = panel.index[core]
        p_ml = pd.DataFrame(np.nan, index=panel.index, columns=["down", "neutral", "up"])
        p_ml.loc[eligible] = clf.predict_proba(x.loc[eligible])
        n = horizon // cfg.grid_minutes
        mean_dyn, variance_dyn = drift_terminal_moments(filtered, dynamics, n)
        band = cfg.neutral_band_sigma * features.sigma_ticks.to_numpy() * np.sqrt(n)
        p_dyn = pd.DataFrame(_gaussian_classes(mean_dyn, np.sqrt(variance_dyn), band), index=panel.index, columns=p_ml.columns)
        p_null = pd.DataFrame(_gaussian_classes(np.zeros(len(panel)), features.sigma_ticks.to_numpy() * np.sqrt(n), band),
                              index=panel.index, columns=p_ml.columns)
        try:
            pool = fit_opinion_pool(p_ml.loc[ca].to_numpy(), p_dyn.loc[ca].to_numpy(), y.loc[ca, "class_id"], wc)
        except ValueError as exc:
            entry.update(status="calibration_failed", reason=str(exc))
            continue
        p_final = pd.DataFrame(opinion_pool(p_ml.to_numpy(), p_dyn.to_numpy(), pool["weight_ml"], pool["temperature"]),
                               index=panel.index, columns=p_ml.columns)
        predictions = pd.DataFrame(index=panel.index)
        for name, probabilities in (("ml", p_ml), ("dynamics", p_dyn), ("pooled", p_final), ("zero_drift", p_null)):
            for j, cls in enumerate(("down", "neutral", "up")):
                predictions[f"{name}_p_{cls}"] = probabilities.iloc[:, j].where(core)
        predictions["endpoint_mean_ticks"] = np.nan
        predictions["mae_long_q80_ticks"] = np.nan
        predictions["mae_short_q80_ticks"] = np.nan
        predictions.loc[eligible, "endpoint_mean_ticks"] = mean_model.predict(x.loc[eligible])
        predictions.loc[eligible, "mae_long_q80_ticks"] = np.maximum(long_risk.predict(x.loc[eligible]), 0)
        predictions.loc[eligible, "mae_short_q80_ticks"] = np.maximum(short_risk.predict(x.loc[eligible]), 0)
        predictions["dynamics_mean_ticks"] = pd.Series(mean_dyn, index=panel.index).where(core)
        predictions["dynamics_std_ticks"] = pd.Series(np.sqrt(variance_dyn), index=panel.index).where(core)
        predictions["indicator"] = (p_final.up - p_final.down).where(core)
        predictions["predictive_entropy"] = (-(p_final * np.log(p_final)).sum(axis=1) / np.log(3)).where(core)
        midpoint = .5 * (p_ml + p_dyn)
        disagreement = .5 * ((p_ml * np.log(p_ml / midpoint)).sum(axis=1) + (p_dyn * np.log(p_dyn / midpoint)).sum(axis=1))
        predictions["expert_js_divergence"] = disagreement.where(core)
        optional = [col for col in columns if col not in FEATURE_MODULES["cgb"]]
        missing_optional = x[optional].isna().any(axis=1) if optional else pd.Series(False, index=x.index)
        predictions["data_status"] = np.where(~panel.cgb_valid, "invalid_current_quote",
            np.where(~core, "insufficient_core_history", np.where(missing_optional, "optional_inputs_missing", "available")))
        predictions["model_use"] = np.where(panel.index <= splits["train"].max(), "in_sample",
            np.where(panel.index <= splits["cal"].max(), "calibration_period", "held_out_or_later"))
        predictions["horizon_eligible_now"] = (panel.index + pd.Timedelta(minutes=horizon) <= panel.session_close)
        unavailable = ~predictions.horizon_eligible_now
        forecast_columns = [col for col in predictions if col not in ("data_status", "model_use", "horizon_eligible_now")]
        predictions.loc[unavailable, forecast_columns] = np.nan
        predictions.loc[unavailable & core, "data_status"] = "outside_session_horizon"
        # Predicting remains possible when the future outcome is not yet observed.
        sessions_test, classes_test = panel.loc[te, "session_id"], y.loc[te, "class_id"].astype(int)
        mean_ml_test = predictions.loc[te, "endpoint_mean_ticks"].to_numpy()
        prior_counts = np.bincount(y.loc[tr, "class_id"].astype(int), weights=wt, minlength=3) + .5
        prior = prior_counts / prior_counts.sum()
        prior_test = np.repeat(prior[None, :], len(te), axis=0)
        # A transparent direction-persistence benchmark with TRAIN-only conditional frequencies.
        past_direction = np.sign(features.mom_z_30).fillna(0).astype(int)
        persistence, persistence_mean = {}, {}
        for sign in (-1, 0, 1):
            mask = past_direction.loc[tr].to_numpy() == sign
            counts = np.bincount(y.loc[tr[mask], "class_id"].astype(int), weights=wt[mask], minlength=3) + .5
            persistence[sign] = counts / counts.sum()
            persistence_mean[sign] = float(np.average(y.loc[tr[mask], "return_ticks"], weights=wt[mask])) if mask.any() else 0.0
        persistence_test = np.array([persistence[int(sign)] for sign in past_direction.loc[te]])
        expert_pairs = {
            "zero_drift": (p_null.loc[te].to_numpy(), np.zeros(len(te))),
            "train_frequency": (prior_test, np.repeat(np.average(y.loc[tr, "return_ticks"], weights=wt), len(te))),
            "past_direction": (persistence_test, np.array([persistence_mean[int(sign)] for sign in past_direction.loc[te]])),
            "dynamics": (p_dyn.loc[te].to_numpy(), predictions.loc[te, "dynamics_mean_ticks"].to_numpy()),
            "ml": (p_ml.loc[te].to_numpy(), mean_ml_test),
            "pooled": (p_final.loc[te].to_numpy(), mean_ml_test),
        }
        metrics = {}
        for name, (probabilities, mean_prediction) in expert_pairs.items():
            metric = _metrics(probabilities, classes_test, y.loc[te, "return_ticks"], mean_prediction, sessions_test)
            metrics[name] = metric
            reports.append({"horizon_minutes": horizon, "expert": name,
                            **{key: metric[key] for key in ("log_loss", "brier", "endpoint_mae_ticks", "rows", "sessions")}})
        risk_report = {}
        for direction in ("long", "short"):
            actual = y.loc[te, f"mae_{direction}_ticks"].to_numpy()
            estimated = predictions.loc[te, f"mae_{direction}_q80_ticks"].to_numpy()
            residual = actual - estimated
            pinball = np.maximum(.8 * residual, -.2 * residual)
            weights_test = _session_weights(sessions_test)
            risk_report[direction] = {"pinball_q80": float(np.average(pinball, weights=weights_test)),
                                      "observed_q80_coverage": float(np.average(actual <= estimated, weights=weights_test))}
        entry.update(status="ready", models={"classifier": clf, "mean": mean_model, "mae_long_q80": long_risk,
                                             "mae_short_q80": short_risk}, pool=pool, predictions=predictions,
                     metrics=metrics, risk_metrics=risk_report, feature_columns=columns,
                     baseline_prior=prior, persistence_probabilities=persistence,
                     notes="Endpoint mean and path quantiles are separate ML heads, not moments of the pooled class probabilities.")
    base["report"] = pd.DataFrame(reports)
    ready = sum(item["status"] == "ready" for item in base["horizons"].values())
    base["status"] = "ready" if ready == len(cfg.horizons_minutes) else ("partial" if ready else "no_eligible_horizons")
    return base


In [14]:
research = fit_research(panel, features, targets, cfg)
print('Research status:', research['status'])
if 'reason' in research:
    display(research['reason'])
if not research['report'].empty:
    display(research['report'])
for h, fitted in research.get('horizons', {}).items():
    print(h, 'minutes:', fitted['status'])
    if fitted['status'] == 'ready':
        display(fitted['risk_metrics'])
        display(fitted['metrics']['pooled']['per_session'])
    else:
        print(fitted.get('reason', ''))


Research status: no_data


### Inspect the latest observation and reuse frozen parameters

The latest row remains the actual latest decision, even when its forecast is unavailable. `predict_frozen` replays newly supplied history with the original fitted parameters. It requires the same configuration and history start. Supply the explicit current cutoff when rebuilding the panel. Forecasts using missing optional features remain marked; trees' ability to accept NaN is not proof that a new missingness pattern is safe.


In [15]:
def latest_readings(research):
    """Latest causal row from each fitted horizon; does not refit or need its label."""
    rows = []
    for horizon, fitted in research.get("horizons", {}).items():
        if fitted.get("status") != "ready":
            rows.append({"horizon_minutes": horizon, "status": fitted.get("status"), "reason": fitted.get("reason")})
            continue
        row = fitted["predictions"].iloc[-1].to_dict()
        unavailable = row["data_status"] in ("invalid_current_quote", "insufficient_core_history", "outside_session_horizon")
        row.update(horizon_minutes=horizon, decision_time=fitted["predictions"].index[-1],
                   status="forecast_unavailable" if unavailable else "research_forecast")
        rows.append(row)
    return pd.DataFrame(rows)


def predict_frozen(research, panel, features, cfg):
    """Replay known history through frozen models; return latest row at each horizon.

    Supply the complete prepared history beginning at filter_history_start, with
    newly received observations appended. Recompute causal features, not fitted
    parameters. A production incremental filter can replace this transparent replay.
    """
    if panel is None or len(panel) == 0 or "dynamics" not in research:
        return pd.DataFrame()
    if panel.index[0] != research["filter_history_start"]:
        raise ValueError("Frozen replay requires history beginning at the original filter_history_start.")
    if cfg != research["config"]:
        raise ValueError("Frozen inference must use the configuration used to train the model.")
    if panel.index[-1] <= research["available_after"]:
        raise ValueError("This fitted model was not available until its calibration outcomes were observed.")
    x = features[research["feature_columns"]].replace([np.inf, -np.inf], np.nan)
    last, now = panel.iloc[-1], panel.index[-1]
    core_good = (bool(last.cgb_valid) and bool(np.isfinite(x.iloc[-1][FEATURE_MODULES["cgb"]]).all())
                 and features.phase_code.iloc[-1] >= 0)
    filtered = filter_local_drift(panel, research["dynamics"]).iloc[[-1]]
    optional = [col for col in x if col not in FEATURE_MODULES["cgb"]]
    data_status = ("invalid_current_quote" if not last.cgb_valid else "insufficient_core_history" if not core_good
                   else "optional_inputs_missing" if x.iloc[-1][optional].isna().any() else "available")
    rows = []
    for horizon, fitted in research["horizons"].items():
        row = {"decision_time": now, "horizon_minutes": horizon, "data_status": data_status,
               "horizon_eligible_now": now + pd.Timedelta(minutes=horizon) <= last.session_close}
        if fitted["status"] != "ready" or not core_good or not row["horizon_eligible_now"]:
            row["status"] = fitted["status"] if fitted["status"] != "ready" else "forecast_unavailable"
            if not row["horizon_eligible_now"]:
                row["data_status"] = "outside_session_horizon"
            rows.append(row)
            continue
        models, pool = fitted["models"], fitted["pool"]
        mean, variance = drift_terminal_moments(filtered, research["dynamics"], horizon // cfg.grid_minutes)
        band = cfg.neutral_band_sigma * features.sigma_ticks.iloc[-1] * np.sqrt(horizon // cfg.grid_minutes)
        p_dyn = _gaussian_classes(mean, np.sqrt(variance), band)
        p_ml = models["classifier"].predict_proba(x.iloc[[-1]])
        p = opinion_pool(p_ml, p_dyn, pool["weight_ml"], pool["temperature"])[0]
        midpoint = .5 * (p_ml[0] + p_dyn[0])
        js = .5 * np.sum(p_ml[0] * np.log(p_ml[0] / midpoint) + p_dyn[0] * np.log(p_dyn[0] / midpoint))
        row.update(status="research_forecast", pooled_p_down=p[0], pooled_p_neutral=p[1], pooled_p_up=p[2],
                   indicator=p[2] - p[0], predictive_entropy=-np.dot(p, np.log(p)) / np.log(3), expert_js_divergence=js,
                   dynamics_mean_ticks=mean[0], dynamics_std_ticks=np.sqrt(variance[0]),
                   endpoint_mean_ticks=float(models["mean"].predict(x.iloc[[-1]])[0]),
                   mae_long_q80_ticks=max(float(models["mae_long_q80"].predict(x.iloc[[-1]])[0]), 0),
                   mae_short_q80_ticks=max(float(models["mae_short_q80"].predict(x.iloc[[-1]])[0]), 0))
        rows.append(row)
    return pd.DataFrame(rows)


In [16]:
live_reading = latest_readings(research)
display(live_reading) if not live_reading.empty else print('No fitted current forecast.')

# When real observations arrive, build a new panel using their explicit cutoff,
# then call predict_frozen(research, new_panel, new_features, cfg).
# Never call fit_research again merely to update the filtered state.


No fitted current forecast.


### Optional scenario inspection

This cell defines a simulator for the fitted drift model. It does not run by default. Simulated paths are conditional Gaussian scenarios, not empirical market paths, executable fills, or an extra source of evidence for the pool.


In [17]:
def simulate_current_path(model, drift_mean, drift_variance, steps, draws=2000, seed=1729):
    """Conditional Gaussian paths in ticks, starting at zero; not observed market paths."""
    if steps < 1 or draws < 1 or drift_variance < 0:
        raise ValueError("steps/draws must be positive and variance nonnegative.")
    rng = np.random.default_rng(seed)
    state = rng.normal(drift_mean, np.sqrt(drift_variance), size=draws)
    paths = np.zeros((draws, steps + 1))
    for step in range(1, steps + 1):
        state = model.phi * state + rng.normal(0, np.sqrt(model.q), size=draws)
        paths[:, step] = paths[:, step - 1] + state + rng.normal(0, np.sqrt(model.r), size=draws)
    return paths


In [18]:
RUN_CONDITIONAL_SCENARIOS = False
scenario_paths = None
if RUN_CONDITIONAL_SCENARIOS and research.get('dynamics') is not None:
    h = 60
    row = live_reading.loc[live_reading.horizon_minutes.eq(h)]
    if len(row) and row.iloc[0]['status'] == 'research_forecast':
        state = research['filtered_state'].iloc[-1]
        scenario_paths = simulate_current_path(research['dynamics'], state.drift_mean,
                                               state.drift_variance, h // cfg.grid_minutes)
        import matplotlib.pyplot as plt
        q = np.quantile(scenario_paths, [.1, .5, .9], axis=0)
        minutes = np.arange(q.shape[1]) * cfg.grid_minutes
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.fill_between(minutes, q[0], q[2], alpha=.2, label='Model scenario 10–90%')
        ax.plot(minutes, q[1], label='Model scenario median')
        ax.set(xlabel='Minutes after decision', ylabel='CGB ticks from current midpoint',
               title='Conditional Gaussian scenarios — not observed market evidence')
        ax.legend(); plt.show()
    else:
        print('No valid current one-hour forecast for scenario inspection.')


## 14. Execution remains separate, but eventual value depends on it

You are right that bond bid/offer is paid in **price**. Yield can describe a signal or approximate risk; it is not the cash amount exchanged at the quote.

For a CGB benchmark buying $Q_c$ contracts at an entry ask and selling at an exit bid,

$$
\Pi^{long}=Q_c\frac{b_x-a_e}{\tau}V-C,
$$

$$
\Pi^{short}=Q_c\frac{b_e-a_x}{\tau}V-C.
$$

Here $V=10$ CAD per tick, and $C$ contains fees and **additional** adverse slippage. The bid/ask crossing is already inside the quote prices. Subtracting another spread would double count it. The tick mechanics follow the [Montréal Exchange CGB specification](https://www.m-x.ca/en/markets/interest-rate-derivatives/cgb).

For a cash bond with face amount $N$ and clean quoted prices per 100 face, the marked cash-price change uses dirty prices:

$$
P^{dirty}=P^{clean}+AI,
$$

$$
\Pi^{long}_{bond}=\frac N{100}
\left(P^{dirty,bid}_x-P^{dirty,ask}_e\right)
+\text{cash flows}-\text{financing}-\text{fees}.
$$

Settlement, accrued interest, coupons, funding and borrow must be supplied consistently for a full ledger. The notebook includes a simple price-touch accounting helper and names these additional cash-flow inputs explicitly. It does not pretend to be a settlement engine.

The rough risk relation

$$
\Delta\Pi\approx-\operatorname{DV01}\,\Delta y_{bp}
$$

helps compare exposure, but an outright cash bond and CGB are not identical positions. Cheapest-to-deliver, conversion factors, basis and key-rate exposures matter. The current notebook forecasts CGB; a cash-bond implementation needs its own exposure mapping and observed price benchmark.

This preserves the distinction you asked for. A market forecast can be right and still be too small to monetize. Execution does not have to define momentum for costs to matter when deciding whether to trade it.


In [19]:
"""Optional price-touch accounting. These functions never fit the market model."""
import math


def _check_touch_prices(entry_bid, entry_ask, exit_bid, exit_ask):
    prices = (entry_bid, entry_ask, exit_bid, exit_ask)
    if not all(math.isfinite(x) and x > 0 for x in prices):
        raise ValueError('Prices must be finite and positive.')
    if entry_bid > entry_ask or exit_bid > exit_ask:
        raise ValueError('Crossed quotes cannot define a price-touch benchmark.')


def futures_touch_benchmark(entry_bid, entry_ask, exit_bid, exit_ask,
                            direction, contracts, cfg, fees_cad=0.0,
                            additional_slippage_ticks_per_contract=0.0):
    """Benchmark one round trip in one fixed CGB contract at supplied quotes.

    Caller must select quotes at actual entry/exit benchmark times, after latency,
    and check freshness, contract identity and sufficient depth. No fill, queue or
    market-impact model is implied. Extra slippage is TOTAL round-trip adverse
    ticks per contract beyond the displayed touches. Spread is already paid once.
    """
    _check_touch_prices(entry_bid, entry_ask, exit_bid, exit_ask)
    if direction not in (-1, 1) or contracts <= 0 or int(contracts) != contracts:
        raise ValueError('direction must be +/-1 and contracts a positive integer.')
    if not all(math.isfinite(x) and x >= 0 for x in (fees_cad, additional_slippage_ticks_per_contract)):
        raise ValueError('Fees and additional adverse slippage must be finite and nonnegative.')
    touch_points = exit_bid - entry_ask if direction == 1 else entry_bid - exit_ask
    mid_points = direction * ((exit_bid + exit_ask) / 2 - (entry_bid + entry_ask) / 2)
    multiplier = contracts * cfg.tick_value_cad / cfg.tick_size
    extra = contracts * additional_slippage_ticks_per_contract * cfg.tick_value_cad
    return {
        'midpoint_change_cad': mid_points * multiplier,
        'spread_crossing_cad': (mid_points - touch_points) * multiplier,
        'touch_change_cad': touch_points * multiplier,
        'additional_slippage_cad': extra, 'fees_cad': fees_cad,
        'net_benchmark_cad': touch_points * multiplier - extra - fees_cad,
        'status': 'price_touch_benchmark_not_verified_fills',
    }


def cash_bond_touch_benchmark(entry_bid_clean, entry_ask_clean,
                              exit_bid_clean, exit_ask_clean,
                              entry_accrued_per_100, exit_accrued_per_100,
                              face_amount, direction, position_cashflows_cad=0.0,
                              financing_and_borrow_cad=0.0, fees_cad=0.0,
                              additional_slippage_cad=0.0):
    """Simple cash-price ledger; clean prices and accrued interest are per 100 face.

    Accrued amounts must match the relevant settlement conventions. Cashflows are
    SIGNED for the actual position, including coupon receipts/payments as needed.
    Financing/borrow and additional adverse slippage are nonnegative costs here.
    This is not a bond valuation, settlement, repo, DV01 or CGB hedge-ratio engine.
    """
    _check_touch_prices(entry_bid_clean, entry_ask_clean, exit_bid_clean, exit_ask_clean)
    amounts = (entry_accrued_per_100, exit_accrued_per_100, face_amount,
               position_cashflows_cad, financing_and_borrow_cad, fees_cad, additional_slippage_cad)
    if not all(math.isfinite(x) for x in amounts) or face_amount <= 0 or direction not in (-1, 1):
        raise ValueError('Finite amounts, positive face and direction +/-1 required.')
    if min(financing_and_borrow_cad, fees_cad, additional_slippage_cad) < 0:
        raise ValueError('Cost inputs cannot be negative in this simple ledger.')
    entry = (entry_ask_clean if direction == 1 else entry_bid_clean) + entry_accrued_per_100
    exit_price = (exit_bid_clean if direction == 1 else exit_ask_clean) + exit_accrued_per_100
    price_change = direction * (exit_price - entry) * face_amount / 100
    return {'dirty_price_change_cad': price_change,
            'position_cashflows_cad': position_cashflows_cad,
            'net_benchmark_cad': price_change + position_cashflows_cad
                - financing_and_borrow_cad - fees_cad - additional_slippage_cad,
            'status': 'price_touch_ledger_not_verified_fills'}


## 15. What is deliberately not smuggled into the implementation

There is no assertion that four days identify a universal swap-flow law. There is no reconstruction of a firm's inventory from its reported trade. There is no global normalization across future data, no backward-smoothed state, no adjustment that turns a contract roll into momentum, and no same-interval confidence sizing.

The baseline does not fit daily EVT, Hurst or Lyapunov machinery to a few intraday sessions and preserve the original names. Their potential usefulness is a research question, but their estimation scales and assumptions would need their own case. In the source code, some of these named quantities are explicitly proxies. Copying a function name is not transferring its mathematical interpretation.

Likewise, SPX and VIX are contextual observations, not fixed-sign trading rules. A risk-off equity move does not compel a duration rally under every inflation and policy environment. Context modules must earn their contribution on matched data.

The notebook contains a full path from supplied canonical observations to fitted forecasts and diagnostics. It does not contain a vendor connection, live order routing, inferred event-book semantics, or measured alpha. Those are concrete boundaries, not unfinished equations hidden behind impressive vocabulary.

## 16. How to use the notebook

Run the cells in order. With inputs left as `None`, it defines the implementation and reports that it is awaiting data. It does not silently substitute an example dataset.

Populate the input cells with the documented quote, rate, session and optional trade tables. Choose a feature-module set before inspecting its test outcome. Begin with the CGB-only version as a baseline, then explicitly compare the duration-environment version on a common eligible period. The short swap history remains a separate research version until its coverage supports the chosen split.

Inspect the coverage and freshness tables before fitting. The fitted result reports each horizon separately, retains the individual experts, and produces a live view only for eligible current observations. A stale latest observation cannot be replaced with an old successful forecast and called live.

Only after the market forecast is understood should the execution helper be used with a stated entry delay, quantity and actual bid/ask observations. Its function is to connect a forecast to a price-based benchmark, not to prove fills or choose a trade automatically.

## 17. Cognitive checklist

- [ ] I can state the target in ticks and distinguish it from net profit.
- [ ] I know which future outcomes the three classes represent.
- [ ] I know the decision time, availability time, forecast horizon and possible entry time.
- [ ] I can identify which transforms are definitions and which parameters are learned.
- [ ] I can explain why common duration and local residual are both retained.
- [ ] I am not treating related curve/forward quantities as independent votes.
- [ ] I can distinguish displayed depth, actual executed flow and an unobserved position.
- [ ] I know whether VWAP came from actual trades and what its band measures.
- [ ] I understand that a descriptive phase can predict any future outcome class.
- [ ] I can explain what the Kalman state measures and what it does not identify.
- [ ] I know why a one-hour and four-hour forecast can disagree.
- [ ] I can say where XGBoost first uses labels and exactly what those labels are.
- [ ] I know which partition fitted every parameter, including combination weights.
- [ ] I can find the simple baseline that could make the elaborate model unnecessary.
- [ ] I am counting independent sessions and events, not just overlapping rows.
- [ ] I can distinguish neutral outcomes, diffuse probabilities and missing information.
- [ ] I have not mistaken scenario simulation or software checks for market evidence.
- [ ] I can account for the spread once, in price, and recognize cash-bond/CGB risk differences.
- [ ] I know which assumptions would fail during a jump, roll, stale feed or changed regime.
- [ ] I can name the next observation that would weaken the model's interpretation.

The intended result is a forecast whose meaning survives inspection: a defined exposure, a causal description, a specified persistence hypothesis, an ML comparison, and a visibly uncertain forecast of an observable future.


## Source and decision record

**Fichiers consultes :** Fractal X `0 README_MODEL.md`, `0.1 AGENTS.md`, agent contracts, notebooks 1–6, and the core PHYSICS essays 01–06; the detailed code map follows in the source-audit appendix. The original repository remains unchanged. The appendix records exact files, zero-based notebook cells and functions.

**Sections ou passages utilises :** exposure/reference construction; distribution and constraint equations; causal timing; stationarity distinctions; metric transforms; current graph tags; multi-horizon XGBoost; structural/Monte Carlo/Born fusion; signal readout and execution timing.

**Logique extraite :** define the environment before the predictor, preserve the reference and its departures, separate observation from future state, and make every probabilistic transformation inspectable.

**Decision deduite :** construct a separate CAD model with available-at snapshots, direct future CGB targets, a filtered drift expert, supervised nonlinear forecasts, an explicit dependent-expert combination and a separate execution ledger.

**Incertitudes ou contradictions :** source claims and implementations are not interchangeable; source state tags and physical quantities require an observation map; some source metrics are named proxies; source backtest timing needs repair before its performance can establish the CAD case; no supplied Canadian data identifies coefficients or predictive value here.

The [gitos-lenses repository](https://github.com/wizzo-gmb/gitos-lenses) supplies research-ordering and evidence-discipline guidance rather than an implementation of this CAD predictor. The [XGBoost parameter documentation](https://xgboost.readthedocs.io/en/stable/parameter.html) describes the probability and quantile objectives used by the implementation. These links are references for the design, not evidence of trading performance.


## Implementation index

Open the [self-contained notebook](cad-duration-momentum.ipynb) to run the design. Its data cells are blank.

| Code entry point | Role |
|---|---|
| `ResearchConfig`, `FEATURE_MODULES` | Editable conventions and explicitly activated measurements |
| `prepare_panel(..., as_of=AS_OF)` | Available-at observations, explicit session grid and current cutoff |
| `build_features` | Causal price, reference, curve, book, VWAP and context transformations |
| `make_targets` | Future price and sampled-path outcomes, excluded from predictors |
| `split_sessions` | Chronological TRAIN / CAL / TEST contract |
| `fit_local_drift`, `filter_local_drift` | Parameter fitting and subsequent forward-only state updates |
| `fit_research` | Three XGBoost horizon families, combination fitting, baselines and reports |
| `latest_readings`, `predict_frozen` | Current output and reuse of already fitted models |
| `simulate_current_path` | Optional conditional Gaussian scenarios |
| `futures_touch_benchmark`, `cash_bond_touch_benchmark` | Separate price-based accounting |

Activated modules are `cgb`, `us`, `cad_futures`, `curve`, `ois`, `swaps`, `forwards`, `book`, `vwap`, and `context`. The default `('cgb',)` is the initial baseline, not a claim that the surrounding market is irrelevant. Enabling a module changes the research version and requires training coverage. `ois` and `swaps` currently require their complete 1y–5y sets when activated; adjust the declared registry before an experiment if the actual universe is narrower.

The endpoint-mean and adverse-excursion outputs are separate regression heads. They are not advertised as moments of a single joint distribution reconstructed from the pooled three-class forecast. The prototype is a transparent batch replay and research indicator, not a production event-processing service.


## Appendix — Exact source audit

# Source-to-model audit for an intraday CAD duration research notebook

Audit date: 2026-09-24. Source root: `C:/Users/mn262/Downloads/fractal-x-main/fractal-x-main`. This is a read-only source inspection; the original models were not executed, packages were not installed, and original files were not changed. Notebook cell numbers below are **zero-based JSON cell indices**, including Markdown cells. The accompanying `source_inventory.json` records source SHA-256 hashes, cells, saved outputs, and 400 function/class definitions with within-cell line numbers.

This document distinguishes source behavior, mathematical interpretation, and proposed CAD extensions. Physical terminology in a comment is not evidence that the corresponding physical quantity has been identified in financial data. A deployable pipeline is not, by itself, demonstrated predictive alpha.

## 1. Source architecture and what the CAD notebook should inherit

The implemented pipeline is:

1. Daily adjusted prices and predefined baskets → inverse-volatility legs and a relative-return leader.
2. Returns → robust rolling moments, fitted tail laws, custom memory/propagation scores, and an engineered GAS/FLUIDE classification.
3. Basket returns → a time-varying reference exposure, residual return, centered level spread, and fragility/cointegration diagnostics.
4. Seven manually specified 22-state transition matrices → validated structural laws.
5. Engineered current-state grammar → future grammar labels → five weighted XGBoost heads → sequential Monte Carlo and structural score fusion → predictions and a separate trading simulation.
6. Saved predictions → a display table with additional current-price filters.

The useful transferable order is **define the exposure and its environment; measure them; construct explicit hypotheses; distinguish current recognition from future prediction; test whether predictions improve the economic target**. The CAD extension should retain outright CGB as the traded exposure. A common-duration factor and a CAD residual are explanatory coordinates; they do not compel a market-neutral spread trade.

### Philosophical passages and their implementation boundary

| Source | Extracted logic | Code connection | CAD decision and limit |
|---|---|---|---|
| `theses/source/PHYSICS/01 beta.md` | Ask which reference exposure is stable under the environment's assumptions. | Notebook 1 baskets; notebook 3 hedge ratio and residual. | Identify outright CGB duration exposure, then distinguish broad duration transmission from local repricing. The source does not supply a CGB-specific beta model. |
| `02 smart beta.md` | Exposure coordinates and crowding can change how a signal behaves. | Fragility, reference spread, regime proxies. | Curve shape, US duration, swaps, and L2 may contextualize CGB pressure. An observed covariance is not identified crowd density. |
| `03 zureck.md` | Effective constraints and stable states can change with the environment. | Hand-engineered state construction and structural law mixture. | Permit observations to contradict the state story. A free-energy decrease is not a theorem that a financial position earns positive PnL. |
| `04 zureck vs laplace.md` | Separate information available at the decision from the subsequent action and outcome; sequential evidence matters. | SPRT-inspired scores and future-state heads. | Use explicit `available_at`, revisions, and future horizon definitions. The financial use of the physical analogy remains a modeling hypothesis. |
| `05 zureck vs laplace part 2.md` | Pointer states, selection by environment, amplitudes, and tail behavior motivate the representation. | Squared positive amplitudes and eleven tags. | The eleven states are engineered in code; they are not derived uniquely from a physical principle. |
| `06 stationary time series.md` | Distinguish return residuals, accumulated levels, reference hedge ratios, and stationarity. | Notebook 3 separate return and level spreads. | Do not conflate residual return, level deviation, stationarity evidence, and the PnL of an executable hedge. |

The physics sources are an interpretable design language. For the new notebook, every financial quantity still needs units, a data construction, a timing rule, and a falsifiable use.

## 2. Notebook 1: reference universe and leader

File: `1 leader univers.ipynb`.

| Cell / function | Input → operation → output | Purpose and caveat |
|---|---|---|
| 16, `REGIME_MAP` | Predefined Q1/Q2 instrument lists → buy/sell baskets. | An imposed environment partition, not an estimated macroeconomic regime from inflation/growth observations. |
| 18, `filter_trading_session`; 28, `fetch_eodhd_daily` | Daily adjusted market data → weekday-filtered price panel. | Weekdays are not an intraday exchange-session calendar. |
| 38, `_inverse_vol_weights_tensor` | Returns → square-root EWMA of squared returns, halflife 63, shifted one row → inverse-volatility weights capped at 0.35 and normalized. | Risk-balancing a basket. The volatility estimate is based on a second moment rather than a demeaned variance. |
| 42, `_rolling_beta_tensor` | Two basket returns → rolling covariance/variance over 63 observations, clipped to [0.05, 3], shifted one row. | A lagged relative exposure estimate with a restrictive positive-beta prior. |
| 52, `build_spread_from_prices` | Log returns and basket weights → $S_t=R_t^+-\beta_tR_t^-$. | This `S` is a **return spread**. It is not the accumulated level spread of notebook 3. |
| 48, `stationarity_gpu` | Return-spread history → simplified Dickey–Fuller-style regression and approximate KPSS diagnostic. | The regression is not a general augmented Dickey–Fuller specification; stationarity of returns is weaker than proving a useful cointegrating price relation. |
| 59, `build_pure_momentum_rotation` | Rolling 126-row sum of spread / rolling spread standard deviation → maximum-score leader. | A relative momentum selector among the prescribed baskets. |
| 62, `backtest_leader_only` | Leader shifted one row × subsequent spread return. | A daily leader-rotation simulation. It does not establish intraday CGB performance. |

## 3. Notebook 2: measured returns, fitted tails, engineered memory

File: `2 metriques.ipynb`. Most windows assume 252 daily rows. Statistical estimation starts here; it is incorrect to say that nothing is fitted until XGBoost.

### 3.1 Returns and moments

Cell 16, `log_returns`, forms $r_t=\log(P_t/P_{t-1})$ and drops missing rows. Dropping rows compresses the observation index: 252 retained observations need not mean 252 exchange sessions or any fixed elapsed time.

Cell 17, `gaussian_roll`, calculates a rolling median and MAD scale, winsorizes each return with its **own** trailing-window threshold, and then computes rolling moments on that historically winsorized series. It is not equivalent to taking one current window and winsorizing all of its observations with the current window's common threshold. The clipping threshold is eight MAD-standardized units. Outputs are mean, sample standard deviation, bias-adjusted skewness, and bias-adjusted excess kurtosis.

These are descriptive features. Their numerical values depend on sampling interval, window length, missing-data treatment, price adjustments, and robustification. They are not instantaneous physical forces.

### 3.2 EVT fitting and hybrid VaR/ES

Cells 21–30 contain `_k_grid`, `_gpd_nll`, `_gpd_fit_mle_scipy`, `_gpd_pwm`, `_ks_gpd_distance`, `_fit_best_gpd_on_sorted`, `_fit_evt_windows`, and `evt_params_from_series`. The left tail is fitted to $-r$ and the right to $r$. Candidate exceedance fractions are approximately 6–20%, with count restrictions; positive exceedances $Y=X-u$ are modeled using a generalized Pareto survival function:

$$
\Pr(Y>y\mid X>u)=\left(1+\xi y/\beta\right)^{-1/\xi},
\qquad \beta>0,\quad 1+\xi y/\beta>0.
$$

Candidate selection combines PWM estimates, MLE refinement, stability, likelihood, and KS-style goodness of fit. It is not simply a universal maximum-likelihood tail estimator.

Cell 31, `evt_quant_es`, maps threshold mass $p_u$ and parameters to an upper-tail magnitude:

$$
q_\alpha=u+\frac{\beta}{\xi}\left[\left(\frac{\alpha}{p_u}\right)^{-\xi}-1\right],
\qquad
\operatorname{ES}_\alpha=q_\alpha+\frac{\beta+\xi(q_\alpha-u)}{1-\xi}.
$$

The $\xi=0$ quantile limit is $u+\beta\log(p_u/\alpha)$; finite ES requires $\xi<1$. The left side is returned with a negative sign. Extrapolation requires the target probability to lie within the fitted tail, not outside its threshold mass.

Cell 56, `hybrid_var_es`, adds empirical quantile/ES fallbacks, shifted and smoothed tail parameters, regime-dependent shape limits, continuity and fit checks, shock exceptions, and monotonicity corrections. These protections make the outputs hybrid estimates, not pure GPD predictions. Cell 57, `ensure_alpha_cols`, sets `alpha_minus` and `alpha_plus` to tail ES quantities and computes

$$
\texttt{alpha\_mu}=\frac{\mu-\alpha(\operatorname{ES}_L+\operatorname{ES}_R)}{1-2\alpha}.
$$

This is a central-body mean identity only when the mean and equal-probability tails form one consistent distributional partition. Mixing winsorized moments, empirical fallbacks, and fitted tails weakens that interpretation. The name `alpha_mu` does not mean investment alpha.

### 3.3 Entropy

Cell 38, `entropy_mixture_series`, uses a return window excluding the current observation and shifted tail parameters. For disjoint left-tail/body/right-tail components, it constructs

$$
h=-\sum_k w_k\log w_k+\sum_k w_k h_k,
\qquad h_{\rm GPD}=\log\beta+1+\xi.
$$

The formula is appropriate to a disjoint-support partition with internally consistent weights. It is not the entropy formula for an arbitrary overlapping mixture. Invalid tail mass is reassigned to the body.

Cell 51, `_gpu_entropy_body_batched`, sorts central observations and uses neighbor/boundary spacings with a digamma correction. Its spacing construction is custom and should be described as a nearest-neighbor-inspired entropy estimator, not automatically certified as a standard textbook estimator. Tied tick returns and a tiny effective body sample are consequential edge cases.

This is **differential entropy**, which depends on return units and can be negative. It should not be compared directly with the discrete maximum entropy $\log 3$. Notebook 5's separate discretized variety feature has a different interpretation.

### 3.4 `rolling_hurst` is a shock-memory proxy

Cell 32, `_expand_qnorm`, rescales a series using expanding 10th/90th percentiles shifted one row, clips to [0, 1], and uses 0.5 on unavailable values.

Cell 33, `rolling_hurst`, does **not** implement a conventional Hurst estimator. It combines normalized tail excess, kurtosis, volatility acceleration, binary-sign entropy defect, curvature, and kinetic activity. Its explicit source score is

$$
U=0.26T+0.22K+0.20A+0.16E+0.16C,
\qquad V=0.54Q+0.24C+0.22T.
$$

It accumulates $U(1-0.42V)$ with decaying memory, mixes the result with source and entropy-defect scores, then maps a bounded energy to $H=0.5+0.5E_H$. The range is [0.5, 1]. It therefore cannot report ordinary anti-persistence $H<0.5$, and it is not estimated from a scaling exponent.

**Timing concern:** a volatility-acceleration intermediate divides by `cp.nanstd(sd)` over the supplied series. Later positive-scale normalization may largely cancel that common factor, but epsilon terms and degeneracy prevent declaring this harmless without a prefix-invariance check. Retain the raw-source issue as a potential future dependence, not a proven large practical effect.

For CAD, either rename this quantity `shock_memory_proxy` and fully disclose its construction, or implement a separately defined conventional estimator with its own assumptions. Do not attach standard Hurst interpretations to this function's name.

### 3.5 `rolling_lyap_kernel` is a propagation proxy

Cell 34, `rolling_lyap_kernel`, builds normalized memory, compression, tail, stress, positive-change fronts, rolling correlations, and lead correlations. Its block combines front movement, target speed, propagation speed, propagation level, memory speed, and memory–receptor interaction, followed by an EWMA and quantile normalization.

It does not estimate exponential separation of nearby trajectories in a reconstructed dynamical system. Describe it as `propagation_instability_proxy`, not a measured Lyapunov exponent. A positive score does not establish deterministic chaos or a prediction horizon.

### 3.6 GAS/FLUIDE and exact timing differences

Cell 35, `classify_gas_fluid_linear_nonlin`, creates two weighted energies. GAS emphasizes body/diffusion/free-flow/calm-tail behavior and weak compression/crowding/memory; FLUIDE emphasizes compression/friction/crowding/tails/collective activity/memory. It reports

$$
p_{\rm gas}=\frac{E_{\rm gas}}{E_{\rm gas}+E_{\rm fluid}},
$$

and chooses the larger-energy regime. This is an engineered score ratio, not a learned regime probability, and it has no inherent long/short direction.

Cell 36, `premice_signals`, combines normalized deviations of short versus long tail/kurtosis summaries and memory/propagation scores. GAS and FLUIDE precursor scores are not exact complements; absolute deviations discard the direction of the deviation.

Cells 59 and 61, `build_M_pit_from_returns` and `build_daily_w252_frame`, require **column-specific** timing:

- Moments and exported returns are shifted one row.
- Exported Hurst proxy is shifted one row.
- Exported Lyapunov proxy is not shifted, so it can use the current completed bar.
- Entropy internally excludes the current return; its GAS input receives another shift.
- Raw versus shifted EVT columns differ; a column-name filter such as cell 58's `pit_only_cols` does not prove causal availability.
- OHLC scaling uses adjusted close divided by the mean of O/H/L, followed by `ffill().bfill()` and clipping. This is not standard corporate-action adjustment; initial backfilling can use later information.

The correct CAD rule is availability at the decision, not a blanket requirement that every feature be shifted exactly one row. Current completed-bar data can be causal if the decision and entry occur afterward.

## 4. Notebook 3: beta, residual, level spread, and fragility

File: `3 beta.ipynb`.

Cells 7–9, `capped_simplex_weights`, `inverse_vol_weights`, and `expanding_hedge_ratio`, estimate inverse-volatility basket weights and expanding covariance/variance ratios using lagged estimates. Cell 13, `compute_smart_spread_pack`, maintains **two distinct quantities**:

$$
R_t^+=\sum_i w_{i,t}^+r_{i,t},\qquad R_t^-=\sum_jw_{j,t}^-r_{j,t},
$$
$$
\alpha_t=R_t^+-\beta_tR_t^-,\qquad
L_t^\pm=\sum_{u\le t}R_u^\pm,\qquad
S_t^{\rm raw}=L_t^+-k_tL_t^-.
$$

Cell 10, `causal_stationary_level_spread`, subtracts a lagged rolling 126-row center from $S^{\rm raw}$. Centering is causal under its input assumptions but **does not prove stationarity or cointegration**. Cells 11, 17, and 18 add lagged standardizations, beta instability, level correlations, and spread deviations.

The level-spread increment is not automatically an executable hedge return:

$$
\Delta S_t^{\rm raw}=R_t^+-k_tR_t^--(k_t-k_{t-1})L_{t-1}^-.
$$

Recentring adds another change term. A portfolio-PnL interpretation needs holdings, rebalance timing, and financing; the code's separate residual-return series avoids that identity error only if users preserve the distinction.

Cell 12, `expanding_fragility_law`, regresses residual return $y_t=\alpha_t$ on its lagged expanding variance $x_t$. Estimates are lagged. The output `sb_alpha_sigma2` is the regression intercept, `sb_fragility_beta` is minus the slope, and `sb_fragility_inertia` divides variance by that negative slope. The last quantity can be negative or unstable near zero. It is not an identified physical relaxation time.

Cells 19–21, `_safe_coint`, `_expanding_coint_stats`, and `_add_ticker_spread_stats`, use Engle–Granger calculations on historical slices, with a minimum 126 rows and periodic recalculation. The special **always recompute the last row** rule means appending data can change which historical timestamps are refit in a full rerun. This is a prefix-consistency concern even if each individual fit only uses its past. Also, cointegrating a cumulative return level with an already centered spread is not automatically the standard two-I(1)-series setting.

Cell 14's ADF/KPSS diagnostic threshold of 0.35 is unconventional; the KPSS implementation's reported p-value range also constrains how that criterion can behave. The diagnostics are not a proof of a stable trading opportunity.

**CAD transfer:** common duration and local residual are useful measured coordinates. A residual need not be forced to be stationary, and the research target remains future outright CGB duration movement. CAD 2/5/10 changes can use the exact basis $L=\Delta y_5$, $s_{25}=\Delta y_5-\Delta y_2$, $s_{510}=\Delta y_{10}-\Delta y_5$; this is a proposed CAD representation, not a source notebook implementation.

## 5. Notebook 4 and persistence bridge

`4 matrices.ipynb`, cell 2, contains seven manually specified 22-by-22 transition matrices and validates row-stochastic structure. Cell 3 persists them and regenerates the loader bridge. `matrices.py`, `load_cell04_matrices`, loads matrices in a prescribed order and rejects missing artifacts.

`fractal_db.py` stores raw market observations using a `DATE` field and stores pipeline run metadata, states, posteriors, splits, and lineage. Relevant functions are `persist_cell04` around line 803 and the cell-05 artifact persistence path around line 1126. The lineage pattern is useful; the daily date schema is not a ready-made intraday market-data model. Choosing the latest successful run independently for different stages can combine incompatible versions unless run identifiers are explicitly reconciled.

## 6. Notebook 5: recognition, learning, and fusion

File: `5 algo fractal x.ipynb`. Important defaults in cell 9 include 252-row memory, horizons 1–5 rows, minimum train/test sizes 400/160, and embargo 7. None of these numbers transfers automatically to a 1–4-hour CGB application.

### 6.1 Current-state construction is engineered recognition

Cell 10 defines eleven tags: R, PTU, SU, U, PEU, EU, PTD, SD, D, PED, ED, duplicated over two environmental blocks for 22 states.

Cell 50, `_regime_motion_context`, residualizes ticker motion against changes in the reference spread, forms multiwindow drift and dispersion, and calculates a Mach-like ratio $|\text{drift}|/(\text{dispersion}+\epsilon)$. It also combines cointegration changes, lagged standardized returns, body-mean changes, compression, memory, and entropy proxies. These are engineered financial coordinates; the Mach analogy does not supply a dimensional physical derivation.

Cell 50, `zureck_tag_p11`, converts positive per-tag amplitudes into $p_j=A_j^2/\sum_k A_k^2$. Cell 53, `_zureck_pointer_filter`, recursively combines emissions, the previous state distribution, and a transition matrix. Positive real amplitudes have no phase interference; mathematically these are weighted score transformations and a recursive state filter.

Cell 47, `zureck_sprt`, uses sequential likelihood-ratio-inspired increments. Its interpretation as a calibrated SPRT requires the assumed null/alternative and dependence model to be appropriate; the code alone does not establish this. Cell 48, `zureck_variety`, discretizes spread behavior into bins; its entropy depends on that binning.

Cells 55–56, `_regime_refine`, `_regime_price_context`, and `build_tags_from_features_strict`, introduce a detailed grammar: prior 20-row breakout boundaries, volatility-scaled buffers, multiple momentum windows, past quantile thresholds, fakeout/rebound conditions, and legal phase sequences. A falling state cannot jump freely to every rising state. This can stabilize interpretation but can also make observed labels reflect the ontology rather than unconstrained market evolution.

Cell 59, `build_transition_graph_tminus1_t`, keeps distinct `graph_tag` (grammar), `graph_observer_tag` (more immediate price evidence), and market-truth diagnostics. It builds node features and adjacent-time edge changes. It is **not a graph neural network**. Cells 103 and 108 add deterministic recognition and 252-row graph memory. The current label is an observation-derived construct, not a forecast and not a trade instruction.

Cell 46, `zfc_block_series`, has a schema hazard: its string fallback uses `pd.factorize(sort=True)`. FLUIDE/GAS map according to the categories present, while other code constructs a block prior in `[p_gas, 1-p_gas]` order. Single-category history can also change the coding. Use an explicit permanent mapping in the CAD notebook; do not assume this fallback is semantically stable.

### 6.2 Feature timing, fitting, targets, and the first supervised ML

Cell 116, `build_dataset`, combines numeric metric/spread features, graph node/edge features, state indicators, and memory. Removing columns containing future/target names cannot prove absence of leakage, and removing direct `p_gas` does not remove the same information encoded in derived states.

Cells 25–29 include expanding past quantiles, shifted z-scores, and a current-inclusive normalizer. A current-inclusive transform can be causal after the current observation is available; it is inappropriate for a decision made before that observation. This must be decided by timestamps, not by the function's name.

Cell 66, `TrainOnlyScaler`, fits mean/std on the outer training set and clips transformed values. Cell 63 supplies chronological splits with cycle-aware boundaries, purge, and embargo. In cell 146, a calibration subset is subsequently taken from the training region. Therefore the scaler has seen the calibration subset's features even though the XGBoost fit uses a narrower fit subset. Affine transforms usually have little effect on trees, but this is not a wholly untouched calibration pipeline. A small-sample fallback can also overlap fit and calibration. Default fractional boundaries move when history grows unless anchor dates are supplied.

Cell 79, `xgb_train_tag`, is the first **supervised machine-learning estimator**. Cell 146, `train_joint7`, trains one `multi:softprob` head per horizon. The key target is the **future engineered `graph_tag`**, shifted by that horizon. Future returns appear separately in audit/evaluation machinery. Predicting a future grammar label accurately does not prove that the label identifies a profitable, persistent CGB move.

Training uses class-balance, transition, horizon, and cybernetic sample weights, plus candidate-selection penalties and calibration-error feedback. In the population limit a weighted cross-entropy classifier estimates a reweighted conditional distribution:

$$
q_w(j\mid x)\propto \mathbb E[w\mid X=x,Y=j]\Pr(Y=j\mid X=x).
$$

Its `predict_proba` output is not automatically a calibrated population probability. It is also **not** a generative likelihood $p(x\mid j)$, despite downstream variables named `L` or `likelihood`. Temperature calibration may improve reliability but does not turn a discriminative posterior into a generative likelihood.

### 6.3 Structural laws and Monte Carlo

Cell 92, `compose_laws`, scores the seven row-wise laws using weighted counts/feature statistics and their log transition values. It applies a softmax-like normalization to those scores and forms a weighted matrix mixture. The weights are fitted structural scores, not automatically a Bayesian posterior over physical laws.

Cell 109, `monte_carlo_leg_propagation`, simulates price paths and maps them through a sequential leg grammar. It uses an EWMA drift/volatility, prior high/low boundaries, at most a limited grammar progression per simulated step, and the current environment block. The first classifier head reweights simulated paths, so the resulting Monte Carlo histogram already carries classifier information.

Two implementation concerns matter for transfer:

- An early volatility fallback uses full-series return standard deviation. This can import future information where that fallback is active.
- The simulator subtracts $\sigma^2/2$ in a GBM log step even though its input mean is estimated from log returns. If that input is interpreted as mean log return, a second Itô correction is inconsistent. The intended drift parameterization must be fixed explicitly rather than copied by name.

The Monte Carlo distribution is conditional on the chosen simulator and grammar; more paths reduce simulation noise, not model misspecification.

### 6.4 Exact active fusion: a tempered product of scores

The active path is cell 145, `born_bayes_posterior_by_horizon`, called from cell 146. It is more complicated than “prior × classifier.” For state $s=(b,j)$, define:

- $q_h(s)$: Monte Carlo tag histogram placed in the current environmental block, with numerical flooring before the amplitude calculation; a grammar projection is the fallback.
- $S_s$: the current-state row of the composed structural matrix, floored positive.
- $F_s$: the frontier weight, floored positive.
- $c_j$: normalized current tag-construction evidence.
- $\ell_h(j)$: weighted XGBoost head output.

Disagreement is measured by the Bhattacharyya overlap and adjusts the diffusion temperature:

$$
B_h=\sum_j\sqrt{c_j\ell_h(j)},\qquad
D_h=\operatorname{clip}\!\left(D_{\rm base}[1+0.5(1-B_h)],1,D_{\max}\right).
$$

With $C_h(s)$ the bounded construction multiplier and $I_{\rm tr}(s)$ a transition-state indicator, the implemented amplitude is

$$
A_h(s)=\big[q_h(s)S_sF_s\big]^{1/(2D_h)}
\,C_h(s)\,\ell_h(j)^{a_h I_{\rm tr}(s)},
\qquad
P_h(s)=\frac{A_h(s)^2}{\sum_u A_h(u)^2}.
$$

Here $C=1$ on persistence states R/U/D; start-state construction multipliers are clipped to [0.5, 2]; finish-state multipliers receive an additional horizon-dependent increase and are clipped to [0.5, 3]. The direct classifier factor acts on transition states, not uniformly on every tag. It is omitted at the first horizon when `mc_carries_signal=True`, because the first head already weighted Monte Carlo paths. Direct-head decay remains at least 0.55 in the ordinary multihead branch.

Consequences:

- Temperature softens the $qSF$ product; it does **not** apply to all factors equally.
- Several factors reuse related inputs, so conditional independence and a coherent joint probability model have not been established.
- The explicit reachability mask is audit-only in this function. It does not force forbidden posterior states to zero. Numerical floors also soften support restrictions. Nevertheless, the engineered labels and Monte Carlo proposal grammar constrain which explanations and paths are readily represented.
- Calling this a **tempered product of expert scores** is a faithful mathematical description. Calling it a derived quantum probability model or ordinary Bayes update is stronger than the implementation establishes.
- Cell 85's older `zureck_born_pnext` is a separate construction that redistributes a tag probability between blocks. Do not substitute that legacy formula for the active multihead path.

Cell 146 later tunes an additional empirical temperature and separately transforms the eleven-tag and 22-state distributions. For temperature unequal to one, independently transforming both need not preserve `tag_probability = sum(state_probabilities over blocks)`. A new implementation should calibrate one coherent distribution and obtain the other by marginalization.

### 6.5 What is genuinely adaptive and what is a diagnostic

Cell 79 changes training weights using calibration errors; cell 115 constructs data-dependent cybernetic weights. Those are actual adaptation mechanisms. By contrast, cell 101's governor `adapt` records diagnostics without changing the strategy on that path, and cell 99's `feedback_degrading` returns false. The philosophical label “cybernetic” spans both active and inert mechanisms; inspect the call path rather than infer behavior from names.

### 6.6 Existing backtest timing is not suitable evidence for the CAD model

Cell 76, `run_trading_backtest`, computes the return from close $i-1$ to close $i$ while sizing with posterior row $i$. Cell 146 passes posteriors without the required shift; those posteriors can incorporate close $i$, current-return temperature, and Monte Carlo starting at close $i$. That creates retrospective sizing of an already realized return. Some pending-entry logic has a delay, but it does not remove this position-sizing issue.

The code also uses generic CFD/FX/crypto cost conventions, not CGB tick-value, contract, depth, and fill conventions. Keep execution and its timing audit separate from the research indicator.

## 7. Notebook 6 and saved evidence

`6 signaux.ipynb`, cell 1, is a display/consumption layer. `read_signal_tags` reads forecasts; `read_graph_tags_t` reads current recognition; `current_signal` selects the latest saved signal. `linreg_stats_on_returns` actually fits a price trend and residual z-score. `build_signal_table` combines forecast tags with price-deviation filters. Date-only formatting loses intraday resolution. It does not fit the model afresh.

No working DuckDB/data/model-output directories were present in the audited checkout. Saved notebook outputs do contain historical results; therefore “there are no results” would be inaccurate. For example, notebook 5 cell 158 includes varied test accuracy/Sharpe logs across equities, IEF, and FX, including negative Sharpe examples. These are saved logs, not independently reproduced results or verified CGB evidence. The source timing issue above further prevents treating those logs as clean proof of predictive alpha.

## 8. Transfer contract for a self-contained 1–4-hour CAD notebook

### Retain, with explicit definitions

1. Exposure/environment separation: outright CGB, common duration, CAD-local residual, and curve shape.
2. A transparent pipeline from observations to measured features to current descriptions to future outcomes.
3. Simple robust moments, explicit return/path summaries, missingness flags, and separate uncertainty.
4. Chronological fit boundaries, a versioned feature contract, and explicit artifact lineage.
5. Current-state diagnostics as interpretable context rather than automatically correct economic labels.

### Replace or defer

1. Replace date-only rows and daily 252/126/63 conventions with elapsed-time/session-aware windows and a minimum-history rule. A month of minute bars is not hundreds of independent daily environments.
2. Define 1/2/4-hour outcomes at actual timestamps. Do not silently truncate a four-hour label at session end or fill missing future observations with unchanged prices.
3. Use `available_at`, revision history, contract identity, and stale-data status. As-of joins must reflect what was known, not merely match exchange timestamps.
4. Keep common-factor exposure and the residual side by side. Do not neutralize away the very duration movement the strategy intends to trade.
5. Learn future CGB movement/path outcomes if that is the economic objective. Predicting the source's next grammar tag is an optional secondary task.
6. Give engineered memory/propagation quantities proxy names. Do not import standard Hurst, Lyapunov, entropy, or quantum claims through naming alone.
7. Defer extreme-tail fitting, high-dimensional states, and elaborate fusion when available history cannot support them. Missing/unestimable is different from neutral/balanced.
8. Separate overlapping mechanism hypotheses—US-led transmission, local repricing, liquidity absorption, and exhaustion need not be mutually exclusive softmax classes.
9. Keep model probabilities distinct from simulator probabilities and execution decisions. Entry latency, bid/ask, depth, costs, and cash PnL belong to an explicitly timed execution module.
10. Leave blank data cells genuinely blank. Definitions and synthetic invariance checks can run without market data; fitted results and performance claims must remain unavailable until real inputs exist.

### Highest-value reference checks

- Prefix invariance: appending future observations must not change historical features that were actually available then.
- Revision/as-of tests: revised yields or late swaps cannot appear in an earlier information set.
- Unit tests in the mathematical sense: basis points versus decimal yields; ticks versus quoted prices; seconds versus rows; log mean versus arithmetic GBM drift.
- Exact curve reconstruction from level and the two slope coordinates.
- Constant prices, zero variance, tied returns, one-category regime history, missing feeds, and interrupted sessions.
- Target separation: a future endpoint/path label never enters features; horizon eligibility is explicit.
- Probability normalization **and** tag/state marginal consistency after any calibration.
- A grammar contradiction can remain visible as evidence rather than being silently relabeled into agreement.

These checks establish that the research object is well defined. They do not establish predictive strength; that requires economically meaningful, chronologically held-out outcomes and comparison with simple persistence/momentum baselines.
